In [ ]:

Logic of this stage:
field → noise → RAG → desire/topography → prompt → writer → convergence → retroaction

In [6]:

# raw_density = 0.5 * audio["syllable_density"] + 0.5 * affect["arousal"]
# density = 1 - exp(-abs(raw_density))

def poetic_saturation(F):
    phon_entropy = audio.get("syllable_density", 0.0)
    repetition   = temporal.get("score_recursive", 0.0)
    affect_var   = affect.get("arousal", 0.0)

def compute_desire(F):
    audio = F["audio_features"]
    affect = F["affect_vector"]
    temporal = F["temporal_features"]

    arousal = affect.get("arousal", 0.0)
    energy  = affect.get("energy", 0.0)
    density = audio.get("syllable_density", 0.0)
    saturation = poetic_saturation(F)

    D = (
        0.4 * arousal +
        0.2 * energy +
        0.2 * density +
        0.2 * saturation
    )

    return np.clip(D, 0.0, 1.0)


def compute_topography(state):
    audio_meta = state["rag_last"]["audio"]["meta"]
    video_meta = state["rag_last"]["video"]["meta"]

    audio_pressure = (
        0.6 * audio_meta.get("rupture_ratio", 0.0) +
        0.4 * audio_meta.get("recursive_score", 0.0)
    )

    video_pressure = (
        0.6 * video_meta.get("rupture_ratio", 0.0) +
        0.4 * video_meta.get("recursive_score", 0.0)
    )

    rag_intensity = min(1.0, len(state["rag_last"]["links"]) / 6.0)

    T = (
        0.4 * audio_pressure +
        0.4 * video_pressure +
        0.2 * rag_intensity
    )

    return np.clip(T, 0.0, 1.0)


def drift_instability(F_hist):
    delta1 = F_hist[-1] - F_hist[-2]
    delta2 = F_hist[-2] - F_hist[-3]
    return norm(delta1 - delta2)


def compute_stability(state):
    D = drift_instability(state["feature_history"])
    R = state["temporal_recursion"]
    U = state.get("agreement_score", 0.0)  # contamination + ambiguity

    S = (
        0.5 * (1 - D) +
        0.3 * R +
        0.2 * U
    )

    return np.clip(S, 0.0, 1.0)


In [7]:

# AUDIO
def collapse_openl3_affect(vectors):
    """
    vectors: List[np.ndarray] (512-d)
    """
    if not vectors:
        return 0.0

    norms = [np.linalg.norm(v) for v in vectors]
    diffs = [
        np.linalg.norm(vectors[i+1] - vectors[i])
        for i in range(len(vectors)-1)
    ] if len(vectors) > 1 else [0.0]

    return float(np.mean(norms) * np.mean(diffs))

def collapse_audio_temporal_structure(audio_analysis):
    return {
        "recursion_score": audio_analysis["score_recursive_drift"],
        "temporal_instability": audio_analysis["score_hybrid"]
    }

def collapse_basic_audio_features(basic_features):
    roughness = (
        basic_features["fricative_density"]
        + basic_features["plosive_density"]
        + (1 - basic_features["vocal_smoothness"])
    ) / 3

    return {
        "silence_ratio": basic_features["silence_ratio"],
        "roughness": roughness,
        "pressure": basic_features["pacing_variance"]
    }

def analyze_audio_for_interrupt(audio_path, affect_vectors, temporal_analysis):
    basic = analyze_audio_features(audio_path)
    
    affect_pressure = collapse_openl3_affect(affect_vectors)
    temporal = collapse_audio_temporal_structure(temporal_analysis)
    physical = collapse_basic_audio_features(basic)

    return {
        "pressure": affect_pressure + physical["pressure"],
        "silence_ratio": physical["silence_ratio"],
        "roughness": physical["roughness"],
        "temporal_instability": temporal["temporal_instability"],
        "recursion_score": temporal["recursion_score"]
    }

# VIDEO

def collapse_clip_affect(vectors):
    if not vectors:
        return 0.0

    diffs = [
        np.linalg.norm(vectors[i+1] - vectors[i])
        for i in range(len(vectors)-1)
    ] if len(vectors) > 1 else [0.0]

    return float(np.mean(diffs))

def collapse_video_temporal_structure(video_analysis):
    return video_analysis["score_recursive_drift"]

def collapse_basic_video_features(mapped_features):
    return {
        "motion_pressure": mapped_features["motion_intensity"],
        "visual_density": (
            mapped_features["edge_dynamics"]
            + mapped_features["frame_density"]
        ) / 2,
        "instability": mapped_features["transition_sharpness"],
        "silence_ratio": mapped_features["visual_silence_ratio"]
    }

def analyze_video_for_interrupt(
    video_path,
    clip_vectors,
    temporal_analysis
):
    mapped = analyze_video_features_mapped(video_path)

    return {
        **collapse_basic_video_features(mapped),
        "recursion_score": collapse_video_temporal_structure(temporal_analysis),
        "instability": collapse_clip_affect(clip_vectors)
    }

def generate_audio_noise(event, text_noise, audio_shards):
    """
    Produces an AUDIO gesture score. No audio synthesis here.
    """

    rng = event.rng
    meta = audio_shards["meta"]

    # --- derive temporal behavior from text geometry ---
    punctuation_density = sum(1 for c in text_noise if c in ".,;:!?") / max(1, len(text_noise))
    # recursion_depth = text_noise.count("[") + text_noise.count("]")
    recursion_depth = (
    text_noise.count("[") +
    text_noise.count("]") +
    text_noise.count("(") +
    text_noise.count(")") +
    text_noise.count("--") +
    text_noise.count("—")
    )

    # --- choose gesture ---
    if meta["rupture_ratio"] > 0.25:
        gesture = "FAIL"
    elif recursion_depth > 3 or meta["recursive_score"] > 2.5:
        gesture = "RECUR"
    elif punctuation_density > 0.08:
        gesture = "DRIFT"
    else:
        gesture = rng.choice(["APPEAR", "SILENCE"])

    # --- temporal shaping ---
    duration = 2.0 + punctuation_density * 10 + recursion_depth * 1.5

    audio_score = {
        "layer": "AUDIO",
        "gesture": gesture,
        "source_file": audio_shards["file"],
        "indices": audio_shards["indices"],
        "chunk_count": len(audio_shards["vectors"]),
        "meta": meta,
        "duration": duration,
        "seed": event.seed
    }

    return audio_score

def generate_video_noise(event, text_noise, video_shards):
    rng = event.rng
    meta = video_shards["meta"]

    punctuation_density = sum(1 for c in text_noise if c in ".,;:!?") / max(1, len(text_noise))
    # recursion_depth = text_noise.count("[") + text_noise.count("]")
    recursion_depth = (
    text_noise.count("[") +
    text_noise.count("]") +
    text_noise.count("(") +
    text_noise.count(")") +
    text_noise.count("--") +
    text_noise.count("—")
    )

    if meta["recursive_score"] > 3.0 or recursion_depth > 4:
        gesture = "RECUR"
    elif meta["rupture_ratio"] > 0.3:
        gesture = "FAIL"
    elif punctuation_density > 0.1:
        gesture = "DRIFT"
    else:
        gesture = rng.choice(["APPEAR", "SILENCE"])

    duration = 3.0 + punctuation_density * 8 + recursion_depth * 2

    return {
        "layer": "VIDEO",
        "gesture": gesture,
        "source_file": video_shards["file"],
        "indices": video_shards["indices"],
        "chunk_count": len(video_shards["vectors"]),
        "meta": meta,
        "duration": duration,
        "seed": event.seed
    }

def enrich_noise_profile(noise_profile, F_vec, F_schema):
    """
    Enrich the noise profile from negotiated text.
    Only structured feature schemas may modulate noise.
    """

    if F_schema != "structured_v1":
        return noise_profile  # incompatible representation

    # --- Reconstruct raw signals from flattened vector ---
    syllable_density = F_vec[0]   # audio_features.syllable_density
    arousal          = F_vec[7]   # affect_vector.arousal
    score_recursive  = F_vec[11]  # temporal_features.score_recursive

    # Raw structural signals (semantic equivalent to original)
    raw_density   = 0.5 * syllable_density + 0.5 * arousal
    raw_recursion = score_recursive

    # Soft compression into (0,1)
    density   = 1.0 - np.exp(-abs(raw_density))
    recursion = 1.0 - np.exp(-abs(raw_recursion))

    # Optional smoothing / attunement (comment out if undesired)
    noise_profile["density"] = (
        0.7 * noise_profile.get("density", 0.0) + 0.3 * float(density)
    )
    noise_profile["recursion"] = (
        0.7 * noise_profile.get("recursion", 0.0) + 0.3 * float(recursion)
    )

    return noise_profile



import random

class NoiseEvent:
    def __init__(self, profile, seed=None):
        self.profile = profile

        if seed is None:
            seed = random.randrange(10**9)

        self.seed = seed
        self.rng = random.Random(seed)

        # Lifecycle flag
        self.enriched = False


import numpy as np

def audio_features_to_timing(low_level_features):
    """
    Maps 7D audio features to temporal shaping parameters.
    Structural only: no semantics.
    """

    # Aggregate over selected chunks
    if not low_level_features:
        return {
            "cut_density": 1.0,
            "speed": 1.0,
            "jitter": 0.0,
            "stutter": 0.0,
            "silence_prob": 0.0
        }

    feats = np.array(low_level_features)

    # Assume columns roughly correspond to:
    # [syllable_density, tempo, pacing_variance, fricative_density, valence, arousal, energy]
    syllable = np.mean(feats[:, 0])
    tempo = np.mean(feats[:, 1])
    pacing_var = np.mean(feats[:, 2])
    fricative = np.mean(feats[:, 3])
    arousal = np.mean(feats[:, 5])
    energy = np.mean(feats[:, 6])

    # Structural mappings
    cut_density = 1.0 + 2.0 * np.tanh(syllable)
    speed = 0.5 + 1.5 * np.tanh(tempo)
    jitter = np.clip(pacing_var, 0.0, 1.0)
    stutter = np.clip(fricative, 0.0, 1.0)
    silence_prob = np.clip(1.0 - energy, 0.0, 1.0)

    return {
        "cut_density": float(cut_density),
        "speed": float(speed),
        "jitter": float(jitter),
        "stutter": float(stutter),
        "silence_prob": float(silence_prob)
    }



def apply_video_drift(filter_chain, timing):
    """
    Adds temporal blur / smear to a video filter chain.
    """
    # Time stretch: slow down or speed up
    drift_factor = 1.0 + (timing.get("jitter", 0.0) - 0.5) * 0.6
    filter_chain += f",setpts={drift_factor}*PTS"

    # Optional motion blur via tmix
    blur_strength = int(1 + timing.get("jitter", 0.0) * 5)
    if blur_strength > 1:
        filter_chain += f",tmix=frames={blur_strength}:weights=1"

    return filter_chain


def apply_audio_drift(filter_chain, timing):
    """
    Adds temporal smear to audio.
    """
    speed = timing.get("speed", 1.0)
    smear = 1.0 + (random.random() - 0.5) * timing.get("jitter", 0.0)

    atempo = max(0.5, min(2.0, speed * smear))
    filter_chain += f",atempo={atempo}"

    # Soft blur via resampling
    filter_chain += ",aresample=44100:resampler=soxr"

    return filter_chain

# Making RECUR → Fractal Nesting

# Instead of simple repetition: A B C  →  A B C A B C
# We create self-embedding: A B C → (A (A B C) B (A B C) C)    Structural recursion in time.

def fractal_sequence(labels, depth=2):
    """
    Builds a recursively nested sequence of labels.
    Pure structure: no semantics.
    """
    if depth <= 1:
        return labels

    out = []
    for l in labels:
        out.append(l)
        out.extend(fractal_sequence(labels, depth - 1))
    return out


# AUDIO & VIDEO FFMPEG GRAPHS & COMMANDS EMITTING FUNCTIONS

def build_audio_ffmpeg_graph(indices, total_chunks, gesture, timing):
    if not indices:
        return None

    n = len(indices)
    duration = timing["duration"]   # <-- FIX HERE

    filters = []
    split_labels = [f"a{i}s" for i in range(n)]
    out_labels = []

    # Split once
    filters.append(f"[0:a]asplit={n}" + "".join(f"[{l}]" for l in split_labels))

    # Trim each chunk
    for i, idx in enumerate(indices):
        start = idx / total_chunks * duration
        end   = (idx + 1) / total_chunks * duration

        # Optional: avoid ugly floats
        start = round(start, 4)
        end   = round(end, 4)

        out_label = f"aud{i}"
        out_labels.append(f"[{out_label}]")

        filters.append(
            f"[{split_labels[i]}]atrim=start={start}:end={end},asetpts=PTS-STARTPTS[{out_label}]"
        )

    # Concatenate
    concat_inputs = "".join(out_labels)
    filters.append(f"{concat_inputs}concat=n={n}:v=0:a=1[outa]")

    return "; ".join(filters)


def build_video_ffmpeg_graph(indices, total_chunks, gesture, timing):
    if not indices:
        return None

    n = len(indices)
    duration = timing["duration"]   # <-- FIX HERE

    filters = []
    split_labels = [f"v{i}s" for i in range(n)]
    out_labels = []

    # Split once
    filters.append(f"[0:v]split={n}" + "".join(f"[{l}]" for l in split_labels))

    # Trim each chunk
    for i, idx in enumerate(indices):
        start = idx / total_chunks * duration
        end   = (idx + 1) / total_chunks * duration

        start = round(start, 4)
        end   = round(end, 4)

        out_label = f"v{i}"
        out_labels.append(f"[{out_label}]")

        filters.append(
            f"[{split_labels[i]}]trim=start={start}:end={end},setpts=PTS-STARTPTS[{out_label}]"
        )

    # Concatenate
    concat_inputs = "".join(out_labels)
    filters.append(f"{concat_inputs}concat=n={n}:v=1:a=0[outv]")

    return "; ".join(filters)

def emit_ffmpeg_commands(event, video_noise, audio_noise, total_chunks, output_prefix="out", media_folder=""):
    cmds = []

    # --- AUDIO ---
    if audio_noise and audio_noise.get("indices"):
        timing = audio_features_to_timing(audio_noise.get("low_level", []))
        timing["duration"] = audio_noise["duration"]   # make sure this is set

        audio_graph = build_audio_ffmpeg_graph(
            indices=audio_noise["indices"],
            total_chunks=total_chunks,
            gesture=audio_noise["gesture"],
            timing=timing
        )

        # 🔍 DEBUG: PRINT AUDIO GRAPH
        print("\n=== AUDIO FILTER GRAPH ===")
        print(audio_graph)
        print("==========================\n")

        if audio_graph:
            audio_file = os.path.join(media_folder, audio_noise['source_file'])
            audio_out = f"{output_prefix}_audio_{event.seed}.wav"
            audio_cmd = (
                f"ffmpeg -y -i \"{audio_file}\" "
                f"-filter_complex \"{audio_graph}\" "
                f"-map \"[outa]\" \"{audio_out}\""
            )
            cmds.append(audio_cmd)

    # --- VIDEO ---
    if video_noise and video_noise.get("indices"):
        timing = {"jitter": 0.5}
        timing["duration"] = video_noise["duration"]   # make sure this is set

        video_graph = build_video_ffmpeg_graph(
            indices=video_noise["indices"],
            total_chunks=total_chunks,
            gesture=video_noise["gesture"],
            timing=timing
        )

        # 🔍 DEBUG: PRINT VIDEO GRAPH
        print("\n=== VIDEO FILTER GRAPH ===")
        print(video_graph)
        print("==========================\n")

        if video_graph:
            video_file = os.path.join(media_folder, video_noise['source_file'])
            video_out = f"{output_prefix}_video_{event.seed}.mp4"
            video_cmd = (
                f"ffmpeg -y -i \"{video_file}\" "
                f"-filter_complex \"{video_graph}\" "
                f"-map \"[outv]\" \"{video_out}\""
            )
            cmds.append(video_cmd)

    return cmds


In [82]:

# DO NOT RUN!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

# Functions we need to import from the previous stage (LET THE NOISE IN)

def sample_corpus(corpus, F_vec, noise_profile, noise_log, k=10):
    noise_load = external_noise_load(noise_log)
    Ft = noisy_target_features(F_vec, noise_profile, noise_load)

    scored = []
    for entry in corpus:
        # F_trans = entry.get("features_trans", {})
        F_trans = entry.get("features_or", {})
        entry_vec = flatten_feature_dict(F_trans)
        entry["features_vec"] = entry_vec  # cache

        d = np.linalg.norm(entry_vec - Ft)
        reuse_penalty = entry.get("recent_hits", 0) * 0.5
        lang_bonus = random.uniform(0, 0.3)
        score = d + reuse_penalty - lang_bonus
        scored.append((score, entry))

    scored.sort(key=lambda x: x[0])
    band_start = random.randint(len(scored)//6, max(len(scored)//3, 1))
    band = scored[band_start:band_start + k]

    for _, e in band:
        e["recent_hits"] = e.get("recent_hits", 0) + 1

    return [e for _, e in band]

def sample_audio_corpus(
    affect_audio_vectors,
    audio_analyses,
    basic_audio_features,
    F_vec,
    event,
    k=3
):
    """
    Returns structurally-selected audio shards based on feature drift,
    rupture density, and event profile. No semantics.
    """

    rng = event.rng

    # --- pick a file deterministically ---
    files = list(affect_audio_vectors.keys())
    file_key = rng.choice(files)

    audio_vecs = affect_audio_vectors[file_key]["vectors"]
    analysis = next(a for a in audio_analyses if a["file"] == file_key)
    low_level = basic_audio_features[files.index(file_key)]

    # --- structural metrics only ---
    rupture_ratio = analysis["ruptures"] / max(1, analysis["segments"])
    recursive_score = analysis["score_recursive_drift"]
    motif_count = analysis["motifs"]

    # --- derive sampling behavior from event profile ---
    density = event.profile.get("density", 0.5)
    recursion_bias = event.profile.get("recursion", 0.5)

    # --- determine number of chunks ---
    n_chunks = min(len(audio_vecs), max(1, int(k + density * 5)))

    # --- choose chunk indices structurally ---
    indices = []
    for _ in range(n_chunks):
        if rng.random() < recursion_bias and analysis["recursive_events"]:
            ev = rng.choice(analysis["recursive_events"])
            idx = int((ev["from"] / analysis["segments"]) * len(audio_vecs))
        elif rng.random() < rupture_ratio:
            rupture_segs = [s for s in analysis["segment_annotations"] if s["type"] == "rupture"]
            seg = rng.choice(rupture_segs)
            idx = int((seg["start"] / analysis["segments"]) * len(audio_vecs))
        else:
            idx = rng.randrange(len(audio_vecs))

        indices.append(max(0, min(idx, len(audio_vecs)-1)))

    # --- collect shards ---
    shards = {
        "file": file_key,
        "indices": indices,
        "vectors": [audio_vecs[i] for i in indices],
        "low_level": [low_level[i] for i in indices if i < len(low_level)],
        "meta": {
            "rupture_ratio": rupture_ratio,
            "recursive_score": recursive_score,
            "motif_count": motif_count
        }
    }

def sample_video_corpus(
    affect_video_vectors,
    video_analyses,
    basic_video_features,
    F_vec,
    event,
    k=3
):
    rng = event.rng

    files = list(affect_video_vectors.keys())
    file_key = rng.choice(files)

    video_vecs = affect_video_vectors[file_key]["vectors"]
    analysis = next(v for v in video_analyses if v["file"] == file_key)
    low_level = basic_video_features[files.index(file_key)]

    rupture_ratio = analysis["ruptures"] / max(1, analysis["segments"])
    recursive_score = analysis["score_recursive_drift"]

    density = event.profile.get("density", 0.5)
    recursion_bias = event.profile.get("recursion", 0.5)

    n_chunks = min(len(video_vecs), max(1, int(k + density * 5)))

    indices = []
    for _ in range(n_chunks):
        if rng.random() < recursion_bias and analysis["recursive_events"]:
            ev = rng.choice(analysis["recursive_events"])
            idx = int((ev["from"] / analysis["segments"]) * len(video_vecs))
        elif rng.random() < rupture_ratio:
            rupture_segs = [s for s in analysis["segment_annotations"] if s["type"] == "rupture"]
            seg = rng.choice(rupture_segs)
            idx = int((seg["start"] / analysis["segments"]) * len(video_vecs))
        else:
            idx = rng.randrange(len(video_vecs))

        indices.append(max(0, min(idx, len(video_vecs)-1)))

    return {
        "file": file_key,
        "indices": indices,
        "vectors": [video_vecs[i] for i in indices],
        "low_level": [low_level[i] for i in indices if i < len(low_level)],
        "meta": {
            "rupture_ratio": rupture_ratio,
            "recursive_score": recursive_score
        }
    }

def sample_links(link_pool, rng, k_min=2, k_max=6):
    n = rng.randint(k_min, k_max)
    return rng.sample(link_pool, min(n, len(link_pool)))



# MEDIA FEATURES FOR THE TASK AT HAND
AudioFeatures = {
    "pressure": float,
    "silence_ratio": float,
    "roughness": float,
    "temporal_instability": float,
    "recursion_score": float
}

VideoFeatures = {
    "motion_pressure": float,
    "visual_density": float,
    "instability": float,
    "silence_ratio": float,
    "recursion_score": float
}

# AUDIO
def collapse_openl3_affect(vectors):
    """
    vectors: List[np.ndarray] (512-d)
    """
    if not vectors:
        return 0.0

    norms = [np.linalg.norm(v) for v in vectors]
    diffs = [
        np.linalg.norm(vectors[i+1] - vectors[i])
        for i in range(len(vectors)-1)
    ] if len(vectors) > 1 else [0.0]

    return float(np.mean(norms) * np.mean(diffs))

def collapse_audio_temporal_structure(audio_analysis):
    return {
        "recursion_score": audio_analysis["score_recursive_drift"],
        "temporal_instability": audio_analysis["score_hybrid"]
    }

def collapse_basic_audio_features(basic_features):
    roughness = (
        basic_features["fricative_density"]
        + basic_features["plosive_density"]
        + (1 - basic_features["vocal_smoothness"])
    ) / 3

    return {
        "silence_ratio": basic_features["silence_ratio"],
        "roughness": roughness,
        "pressure": basic_features["pacing_variance"]
    }

def analyze_audio_for_interrupt(audio_path, affect_vectors, temporal_analysis):
    basic = analyze_audio_features(audio_path)
    
    affect_pressure = collapse_openl3_affect(affect_vectors)
    temporal = collapse_audio_temporal_structure(temporal_analysis)
    physical = collapse_basic_audio_features(basic)

    return {
        "pressure": affect_pressure + physical["pressure"],
        "silence_ratio": physical["silence_ratio"],
        "roughness": physical["roughness"],
        "temporal_instability": temporal["temporal_instability"],
        "recursion_score": temporal["recursion_score"]
    }

# VIDEO

def collapse_clip_affect(vectors):
    if not vectors:
        return 0.0

    diffs = [
        np.linalg.norm(vectors[i+1] - vectors[i])
        for i in range(len(vectors)-1)
    ] if len(vectors) > 1 else [0.0]

    return float(np.mean(diffs))

def collapse_video_temporal_structure(video_analysis):
    return video_analysis["score_recursive_drift"]

def collapse_basic_video_features(mapped_features):
    return {
        "motion_pressure": mapped_features["motion_intensity"],
        "visual_density": (
            mapped_features["edge_dynamics"]
            + mapped_features["frame_density"]
        ) / 2,
        "instability": mapped_features["transition_sharpness"],
        "silence_ratio": mapped_features["visual_silence_ratio"]
    }

def analyze_video_for_interrupt(
    video_path,
    clip_vectors,
    temporal_analysis
):
    mapped = analyze_video_features_mapped(video_path)

    return {
        **collapse_basic_video_features(mapped),
        "recursion_score": collapse_video_temporal_structure(temporal_analysis),
        "instability": collapse_clip_affect(clip_vectors)
    }



# AUDIO SAMPLER & NOISE GENERATOR
def sample_audio_corpus(
    affect_audio_vectors,
    audio_analyses,
    basic_audio_features,
    F_vec,
    event,
    k=3
):
    """
    Returns structurally-selected audio shards based on feature drift,
    rupture density, and event profile. No semantics.
    """

    rng = event.rng

    # --- pick a file deterministically ---
    files = list(affect_audio_vectors.keys())
    file_key = rng.choice(files)

    audio_vecs = affect_audio_vectors[file_key]["vectors"]
    analysis = next(a for a in audio_analyses if a["file"] == file_key)
    low_level = basic_audio_features[files.index(file_key)]

    # --- structural metrics only ---
    rupture_ratio = analysis["ruptures"] / max(1, analysis["segments"])
    recursive_score = analysis["score_recursive_drift"]
    motif_count = analysis["motifs"]

    # --- derive sampling behavior from event profile ---
    density = event.profile.get("density", 0.5)
    recursion_bias = event.profile.get("recursion", 0.5)

    # --- determine number of chunks ---
    n_chunks = min(len(audio_vecs), max(1, int(k + density * 5)))

    # --- choose chunk indices structurally ---
    indices = []
    for _ in range(n_chunks):
        if rng.random() < recursion_bias and analysis["recursive_events"]:
            ev = rng.choice(analysis["recursive_events"])
            idx = int((ev["from"] / analysis["segments"]) * len(audio_vecs))
        elif rng.random() < rupture_ratio:
            rupture_segs = [s for s in analysis["segment_annotations"] if s["type"] == "rupture"]
            seg = rng.choice(rupture_segs)
            idx = int((seg["start"] / analysis["segments"]) * len(audio_vecs))
        else:
            idx = rng.randrange(len(audio_vecs))

        indices.append(max(0, min(idx, len(audio_vecs)-1)))

    # --- collect shards ---
    shards = {
        "file": file_key,
        "indices": indices,
        "vectors": [audio_vecs[i] for i in indices],
        "low_level": [low_level[i] for i in indices if i < len(low_level)],
        "meta": {
            "rupture_ratio": rupture_ratio,
            "recursive_score": recursive_score,
            "motif_count": motif_count
        }
    }

    return shards

def generate_audio_noise(event, text_noise, audio_shards):
    """
    Produces an AUDIO gesture score. No audio synthesis here.
    """

    rng = event.rng
    meta = audio_shards["meta"]

    # --- derive temporal behavior from text geometry ---
    punctuation_density = sum(1 for c in text_noise if c in ".,;:!?") / max(1, len(text_noise))
    # recursion_depth = text_noise.count("[") + text_noise.count("]")
    recursion_depth = (
    text_noise.count("[") +
    text_noise.count("]") +
    text_noise.count("(") +
    text_noise.count(")")
    )

    # --- choose gesture ---
    if meta["rupture_ratio"] > 0.25:
        gesture = "FAIL"
    elif recursion_depth > 3 or meta["recursive_score"] > 2.5:
        gesture = "RECUR"
    elif punctuation_density > 0.08:
        gesture = "DRIFT"
    else:
        gesture = rng.choice(["APPEAR", "SILENCE"])

    # --- temporal shaping ---
    duration = 2.0 + punctuation_density * 10 + recursion_depth * 1.5

    audio_score = {
        "layer": "AUDIO",
        "gesture": gesture,
        "source_file": audio_shards["file"],
        "indices": audio_shards["indices"],
        "chunk_count": len(audio_shards["vectors"]),
        "meta": meta,
        "duration": duration,
        "seed": event.seed
    }

    return audio_score

# VIDEO SAMPLER & NOISE GENERATOR
def sample_video_corpus(
    affect_video_vectors,
    video_analyses,
    basic_video_features,
    F_vec,
    event,
    k=3
):
    rng = event.rng

    files = list(affect_video_vectors.keys())
    file_key = rng.choice(files)

    video_vecs = affect_video_vectors[file_key]["vectors"]
    analysis = next(v for v in video_analyses if v["file"] == file_key)
    low_level = basic_video_features[files.index(file_key)]

    rupture_ratio = analysis["ruptures"] / max(1, analysis["segments"])
    recursive_score = analysis["score_recursive_drift"]

    density = event.profile.get("density", 0.5)
    recursion_bias = event.profile.get("recursion", 0.5)

    n_chunks = min(len(video_vecs), max(1, int(k + density * 5)))

    indices = []
    for _ in range(n_chunks):
        if rng.random() < recursion_bias and analysis["recursive_events"]:
            ev = rng.choice(analysis["recursive_events"])
            idx = int((ev["from"] / analysis["segments"]) * len(video_vecs))
        elif rng.random() < rupture_ratio:
            rupture_segs = [s for s in analysis["segment_annotations"] if s["type"] == "rupture"]
            seg = rng.choice(rupture_segs)
            idx = int((seg["start"] / analysis["segments"]) * len(video_vecs))
        else:
            idx = rng.randrange(len(video_vecs))

        indices.append(max(0, min(idx, len(video_vecs)-1)))

    return {
        "file": file_key,
        "indices": indices,
        "vectors": [video_vecs[i] for i in indices],
        "low_level": [low_level[i] for i in indices if i < len(low_level)],
        "meta": {
            "rupture_ratio": rupture_ratio,
            "recursive_score": recursive_score
        }
    }

def generate_video_noise(event, text_noise, video_shards):
    rng = event.rng
    meta = video_shards["meta"]

    punctuation_density = sum(1 for c in text_noise if c in ".,;:!?") / max(1, len(text_noise))
    # recursion_depth = text_noise.count("[") + text_noise.count("]")
    recursion_depth = (
    text_noise.count("[") +
    text_noise.count("]") +
    text_noise.count("(") +
    text_noise.count(")") +
    text_noise.count("--") +
    text_noise.count("—")
    )

    if meta["recursive_score"] > 3.0 or recursion_depth > 4:
        gesture = "RECUR"
    elif meta["rupture_ratio"] > 0.3:
        gesture = "FAIL"
    elif punctuation_density > 0.1:
        gesture = "DRIFT"
    else:
        gesture = rng.choice(["APPEAR", "SILENCE"])

    duration = 3.0 + punctuation_density * 8 + recursion_depth * 2

    return {
        "layer": "VIDEO",
        "gesture": gesture,
        "source_file": video_shards["file"],
        "indices": video_shards["indices"],
        "chunk_count": len(video_shards["vectors"]),
        "meta": meta,
        "duration": duration,
        "seed": event.seed
    }




In [ ]:

We now introduce D (desire) and T (topography) in three places:

how noisy the target is,

how we score distance,

how wide / erratic the selection band is.

In [ ]:
This means:

when the poem is erotic + urban,
→ noise becomes heavy-tailed, unlearnable, discontinuous;

when the poem is structurally coherent,
→ noise becomes metabolizable again.

In [ ]:


We treat topography (T) as:

density (crowds, compression),

rupture (cuts, jumps, interruptions),

circulation (flows, movement),

saturation (noise, glare, overlap).

This does not add meaning; it adds pressure on form.

In [ ]:

# STATE 

In [ ]:

# Aesthetic Agreement: When Does the Field “Settle”?


In [79]:

def aesthetic_agreement(state, profile, prev_state=None, wD=0.3, wT=0.3, wR=0.1, wE=0.1, wS=0.2):
    D = state["desire"]["arousal"] + state["desire"]["excess"]
    T = state["topography"]
    S = state["stability"]

    S_prev = prev_state["stability"] if prev_state else S

    rupture = profile["rupture_scalar"]
    entropy = profile.get("register_entropy", 0.0)

    A = (
        wD * D +
        wT * T +
        wR * rupture +
        wE * entropy -
        wS * abs(S - S_prev)
    )
    return A


In [ ]:

# Convergence as Negotiation (poet ↔ system)


In [93]:


def fuse(A, B, state):
    return f"{A}\n{B}"

def rewrite(text_A, text_B, states, noise_field):
    erotic = states[-1]["erotic_vector"]
    erotic_prev = states[-2]["erotic_vector"]
    # if role == "poet":
        # weight = 0.7 + 0.6 * erotic["intensity"]
    # else:
    weight = 1.0 + 0.8 * erotic_prev["intensity"] + 0.4 * erotic["volatility"]

    return f"{text_B} ~ {text_A[:int(30 * weight)]}"


In [ ]:

# Temporal recursion + agreement


In [ ]:

The Whole Machine

Multimodal sampling where:

audio/video embody urban pressure,

text, sound, and image are expressions of one state vector.

Aesthetic agreement defined by:

desire (erotic, affective force),

topography (city, rupture, density),

stability (memory, recognizability).

Temporal recursion where:

the future rewrites the past only when the field is alive.

=> distributed subjectivity where city, body, memory, and algorithm negotiate what counts as a poem...

In [ ]:

Higher Desire (D) → stronger rewriting, heavier-tailed noise, more semantic drift.

Higher Topography (T) → more rupture in audio/video, harsher montage.

Lower Stability (S) → deeper temporal recursion (history rewritten).

The poet does not “edit lines.” She reprograms the weather in which the poem grows.

In [ ]:

# Erotic Force as a Vector in the Field


In [47]:

# THE MODELS


models = ["gpt-4.1", "qwen2.5-7b"]

def call_writer_model(prompt, model):
    if model == "gpt-4.1":
        return call_gpt41(prompt)
    elif model == "qwen2.5-7b":
        return call_qwen(prompt)
    else:
        raise ValueError(f"Unknown model: {model}")

from huggingface_hub import InferenceClient
import os

# HF_TOKEN = os.environ.get("HF_TOKEN")  # safer than []

client_qwen = InferenceClient(
    # model="Qwen/Qwen2.5-14B-Instruct",
    model="Qwen/Qwen2.5-7B-Instruct",
    token=YOUR_TOKEN
)

def call_qwen(prompt):
    try:
        response = client_qwen.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            temperature=0.9,
            max_tokens=1000,
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"<<QWEN FAILED: {str(e)[:120]}>>"

def call_gpt41(prompt):
    try:
        response = client.responses.create(
            model="gpt-4.1",
            input=prompt
        )
        return response.output_text
    except Exception as e:
        return f"<<GPT-4.1 FAILED: {str(e)[:120]}>>"

from openai import OpenAI
import os

client = OpenAI(api_key=YOUR_KEY) 


In [10]:

# media logs from previous stage # REPLACE W YOURS
raw_media_logs = [
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 857750491
[AUDIO]  gesture=RECUR file=I_de_linfinit_a_ella_Styli_locus_2020_Mar_Puchol_Foz.mp4 indices=[52, 15, 8, 30, 24, 44, 5] duration=8.12
[VIDEO]  gesture=FAIL file=Labor_inacabada_Trptic_de_la_terra_2020_Merc_Ibarz_Ibarz.mp4 indices=[10, 20, 17, 21, 37, 12, 12] duration=11.10
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 827066999
[AUDIO]  gesture=RECUR file=JV_Foix__EN_VEU_ALTA_Es_quan_dormo_que_hi_veig_clar.mp4 indices=[19, 25, 9, 3, 22, 15] duration=14.16
[VIDEO]  gesture=RECUR file=JV_FOIX__Fou_diumenge_passat_a_les_tres_de_la_tarda.mp4 indices=[23, 27, 0, 22, 34, 2] duration=19.13
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 657990430
[AUDIO]  gesture=RECUR file=La_Marieta_Encara_rai_Les_vint-i-una_falria_2002_Mary_Zapater_Labrador.mp4 indices=[52, 45, 122, 126, 34, 26] duration=2.25
[VIDEO]  gesture=SILENCE file=Seda_de_Josep_Porcar.mp4 indices=[18, 20, 0, 17, 12, 12] duration=3.20
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 557729655
[AUDIO]  gesture=RECUR file=J_Porcar_Nocturn.mp4 indices=[52, 66, 21, 60, 72, 5] duration=5.11
[VIDEO]  gesture=SILENCE file=I_de_linfinit_a_ella_Styli_locus_2020_Mar_Puchol_Foz.mp4 indices=[58, 0, 1, 55, 41, 11] duration=7.08
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 369028707
[AUDIO]  gesture=RECUR file=Flor_de_Sac_Relats_criminals_III-IV_El_vifa_sang_2019_Pilar_Arbiol_Sagarra.mp4 indices=[26, 37, 46, 39, 9, 35] duration=2.00
[VIDEO]  gesture=FAIL file=Una_rosa.mp4 indices=[33, 27, 14, 41, 45, 29] duration=3.00
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 251253279
[AUDIO]  gesture=FAIL file=Cant_a_la_terra_Lletres_captives_Memor_Terra_1999_Maria_del_Carme_Alcover_Pins.mp4 indices=[48, 5, 52, 25, 8, 9] duration=2.00
[VIDEO]  gesture=APPEAR file=Lencs_del_foc_Te_de_roca_2000_Glria_Francino_Pinasa.mp4 indices=[24, 80, 48, 20, 83, 29] duration=3.00
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 388924847
[AUDIO]  gesture=RECUR file=JV_Foix__EN_VEU_ALTA_Es_quan_dormo_que_hi_veig_clar.mp4 indices=[25, 30, 25, 3, 20, 27] duration=14.19
[VIDEO]  gesture=RECUR file=Lencs_del_foc_Te_de_roca_2000_Glria_Francino_Pinasa.mp4 indices=[102, 21, 1, 47, 86, 21] duration=19.15
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 395405288
[AUDIO]  gesture=RECUR file=Eres_tu_Plana_rasa_50_2018_Carmeta_Pallars_Soro.mp4 indices=[12, 2, 15, 9, 3, 31] duration=2.11
[VIDEO]  gesture=FAIL file=Caminant_escolta_Eixam_de_poemes_2010_Teresa_Jass_y_Cas.mp4 indices=[24, 7, 19, 29, 37, 26] duration=3.09
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 832802915
[AUDIO]  gesture=RECUR file=Joan_Salvat_Papasseit_Venedor_damor.mp4 indices=[33, 43, 39, 89, 29, 91] duration=2.35
[VIDEO]  gesture=FAIL file=Flor_de_Sac_Relats_criminals_III-IV_El_vifa_sang_2019_Pilar_Arbiol_Sagarra.mp4 indices=[42, 13, 61, 59, 23, 4] duration=3.28
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 274353247
[AUDIO]  gesture=RECUR file=Caminant_escolta_Eixam_de_poemes_2010_Teresa_Jass_y_Cas.mp4 indices=[26, 9, 34, 33, 31, 18, 1] duration=2.08
[VIDEO]  gesture=RECUR file=Jim_de_Mag_Sunyer.mp4 indices=[5, 131, 95, 105, 121, 76, 42] duration=3.06
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 332191847
[AUDIO]  gesture=FAIL file=Cant_a_la_terra_Lletres_captives_Memor_Terra_1999_Maria_del_Carme_Alcover_Pins.mp4 indices=[52, 24, 26, 50, 6, 52] duration=5.19
[VIDEO]  gesture=FAIL file=Poemes_de_Gabriel_Guasch_Escric_un_cercle.mp4 indices=[0, 0, 13, 13, 10, 13] duration=7.15
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 33704689
[AUDIO]  gesture=RECUR file=Citologia_de_Josep_Porcar.mp4 indices=[30, 12, 57, 61, 6, 7] duration=5.16
[VIDEO]  gesture=SILENCE file=Eres_tu_Plana_rasa_50_2018_Carmeta_Pallars_Soro.mp4 indices=[36, 21, 21, 26, 11, 34] duration=7.13
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 185848096
[AUDIO]  gesture=RECUR file=Citologia_de_Josep_Porcar.mp4 indices=[89, 43, 41, 89, 82, 8] duration=2.15
[VIDEO]  gesture=SILENCE file=Cer_avant_Dona_lletra_aigua_indit_Marta_Momblant_Ribas.mp4 indices=[43, 80, 50, 66, 66, 10] duration=3.12
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 791703375
[AUDIO]  gesture=RECUR file=JV_Foix__En_Veu_Alta__Sol_i_de_dol_Si_pogus_acordar_ra_i_follia.mp4 indices=[7, 22, 87, 87, 65, 57] duration=17.26
[VIDEO]  gesture=RECUR file=JV_FOIX__Fou_diumenge_passat_a_les_tres_de_la_tarda.mp4 indices=[5, 6, 20, 24, 1, 19] duration=23.21
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 605201359
[AUDIO]  gesture=RECUR file=Poema_7_IV_monstruositat_Esclat_2018_Merxe_Llop_Alfonso.mp4 indices=[47, 23, 36, 38, 24, 22] duration=5.20
[VIDEO]  gesture=RECUR file=Citologia_de_Josep_Porcar.mp4 indices=[12, 86, 51, 65, 24, 74] duration=7.16
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 774061355
[AUDIO]  gesture=RECUR file=Poema_7_IV_monstruositat_Esclat_2018_Merxe_Llop_Alfonso.mp4 indices=[12, 33, 32, 37, 34, 43] duration=8.22
[VIDEO]  gesture=APPEAR file=Cant_a_la_terra_Lletres_captives_Memor_Terra_1999_Maria_del_Carme_Alcover_Pins.mp4 indices=[27, 0, 51, 53, 13, 18] duration=11.18
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 962743181
[AUDIO]  gesture=RECUR file=El_batec_del_temps_2015_Cari_Ario_Freja.mp4 indices=[97, 97, 52, 13, 18, 31, 54] duration=2.11
[VIDEO]  gesture=FAIL file=Poema_7_IV_monstruositat_Esclat_2018_Merxe_Llop_Alfonso.mp4 indices=[50, 0, 40, 49, 0, 48, 30] duration=3.09
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 384374115
[AUDIO]  gesture=RECUR file=Citologia_de_Josep_Porcar.mp4 indices=[35, 76, 89, 52, 67, 3] duration=5.25
[VIDEO]  gesture=APPEAR file=Cant_a_la_terra_Lletres_captives_Memor_Terra_1999_Maria_del_Carme_Alcover_Pins.mp4 indices=[20, 7, 20, 40, 12, 42] duration=7.20
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 84922440
[AUDIO]  gesture=RECUR file=Present_poema_de_Mag_Sunyer.mp4 indices=[24, 8, 24, 19, 24, 1] duration=5.10
[VIDEO]  gesture=RECUR file=Una_frustraci_La_mostra_de_lolivera_1995_Maria_Pilar_Febas_Fornos.mp4 indices=[54, 6, 72, 34, 89, 36] duration=7.08
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 507354885
[AUDIO]  gesture=RECUR file=Lencs_del_foc_Te_de_roca_2000_Glria_Francino_Pinasa.mp4 indices=[17, 53, 29, 90, 66, 29] duration=5.14
[VIDEO]  gesture=RECUR file=Una_frustraci_La_mostra_de_lolivera_1995_Maria_Pilar_Febas_Fornos.mp4 indices=[14, 88, 32, 56, 96, 41] duration=7.11
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 443781784
[AUDIO]  gesture=APPEAR file=Sol_i_de_dol.mp4 indices=[12, 61, 77, 58, 58, 4] duration=5.00
[VIDEO]  gesture=APPEAR file=Eres_tu_Plana_rasa_50_2018_Carmeta_Pallars_Soro.mp4 indices=[4, 14, 12, 33, 15, 27] duration=7.00
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 85668015
[AUDIO]  gesture=RECUR file=Lencs_del_foc_Te_de_roca_2000_Glria_Francino_Pinasa.mp4 indices=[100, 18, 64, 0, 100, 78] duration=2.07
[VIDEO]  gesture=SILENCE file=Salvador_Espriu_En_Veu_Alta_Sentit_a_la_manera_de_Salvador_Espriu.mp4 indices=[31, 33, 11, 27, 28, 5] duration=3.06
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 383667712
[AUDIO]  gesture=RECUR file=Flor_de_Sac_Relats_criminals_III-IV_El_vifa_sang_2019_Pilar_Arbiol_Sagarra.mp4 indices=[16, 14, 59, 29, 36, 31, 59] duration=11.05
[VIDEO]  gesture=RECUR file=Present_poema_de_Mag_Sunyer.mp4 indices=[10, 5, 0, 15, 13, 0, 20] duration=15.04
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 664256496
[AUDIO]  gesture=RECUR file=Poema_7_IV_monstruositat_Esclat_2018_Merxe_Llop_Alfonso.mp4 indices=[43, 19, 16, 29, 5, 5] duration=9.56
[VIDEO]  gesture=RECUR file=Salvador_Espriu_En_Veu_Alta_Sentit_a_la_manera_de_Salvador_Espriu.mp4 indices=[12, 4, 0, 19, 3, 33] duration=13.05
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 147506461
[AUDIO]  gesture=FAIL file=Cant_a_la_terra_Lletres_captives_Memor_Terra_1999_Maria_del_Carme_Alcover_Pins.mp4 indices=[52, 29, 21, 14, 52, 24, 21] duration=8.00
[VIDEO]  gesture=RECUR file=El_batec_del_temps_2015_Cari_Ario_Freja.mp4 indices=[20, 59, 3, 67, 37, 99, 3] duration=11.00
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 91062732
[AUDIO]  gesture=RECUR file=Joan_Salvat_Papasseit_Venedor_damor.mp4 indices=[26, 8, 0, 76, 47, 69] duration=8.03
[VIDEO]  gesture=FAIL file=Una_rosa.mp4 indices=[45, 13, 0, 45, 19, 21] duration=11.03
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 844112830
[AUDIO]  gesture=RECUR file=El_trencaclosques_2020_Slvia_Ferragut_Borbn.mp4 indices=[31, 21, 76, 4, 5, 65] duration=2.18
[VIDEO]  gesture=RECUR file=La_Marieta_Encara_rai_Les_vint-i-una_falria_2002_Mary_Zapater_Labrador.mp4 indices=[101, 127, 41, 90, 56, 57] duration=3.14
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 794130383
[AUDIO]  gesture=RECUR file=JV_Foix__EN_VEU_ALTA_Es_quan_dormo_que_hi_veig_clar.mp4 indices=[14, 14, 3, 13, 3, 25] duration=2.13
[VIDEO]  gesture=APPEAR file=Salvador_Espriu_En_Veu_Alta_Sentit_a_la_manera_de_Salvador_Espriu.mp4 indices=[4, 7, 12, 20, 27, 15] duration=3.10
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 158362120
[AUDIO]  gesture=RECUR file=JV_Foix__En_Veu_Alta__Sol_i_de_dol_Si_pogus_acordar_ra_i_follia.mp4 indices=[21, 87, 54, 45, 87, 87] duration=2.13
[VIDEO]  gesture=FAIL file=La_iaia_Maria_Roda_la_mola_2010_Aurlia_Lombarte_Segura.mp4 indices=[5, 9, 45, 41, 20, 23] duration=3.10
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 859437394
[AUDIO]  gesture=RECUR file=Cer_avant_Dona_lletra_aigua_indit_Marta_Momblant_Ribas.mp4 indices=[106, 83, 61, 57, 62, 66] duration=8.16
[VIDEO]  gesture=FAIL file=El_trencaclosques_2020_Slvia_Ferragut_Borbn.mp4 indices=[13, 30, 70, 76, 81, 37] duration=11.13
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 898391860
[AUDIO]  gesture=RECUR file=Lencs_del_foc_Te_de_roca_2000_Glria_Francino_Pinasa.mp4 indices=[31, 63, 21, 21, 1, 41] duration=8.17
[VIDEO]  gesture=FAIL file=Present_poema_de_Mag_Sunyer.mp4 indices=[10, 15, 17, 24, 12, 7] duration=11.14
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 716269599
[AUDIO]  gesture=RECUR file=I_de_linfinit_a_ella_Styli_locus_2020_Mar_Puchol_Foz.mp4 indices=[41, 41, 19, 33, 39, 43] duration=3.66
[VIDEO]  gesture=FAIL file=Flor_de_Sac_Relats_criminals_III-IV_El_vifa_sang_2019_Pilar_Arbiol_Sagarra.mp4 indices=[0, 0, 39, 58, 4, 27] duration=5.13
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 3962740
[AUDIO]  gesture=RECUR file=JV_Foix__EN_VEU_ALTA_Es_quan_dormo_que_hi_veig_clar.mp4 indices=[34, 16, 14, 34, 2, 34] duration=2.19
[VIDEO]  gesture=SILENCE file=Cer_avant_Dona_lletra_aigua_indit_Marta_Momblant_Ribas.mp4 indices=[55, 90, 51, 67, 85, 104] duration=3.15
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 972975344
[AUDIO]  gesture=APPEAR file=90_dAfers_Domstics.mp4 indices=[113, 3, 6, 23, 117, 128, 57] duration=6.65
[VIDEO]  gesture=SILENCE file=Eres_tu_Plana_rasa_50_2018_Carmeta_Pallars_Soro.mp4 indices=[8, 16, 15, 9, 22, 9, 5] duration=9.12
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 349985052
[AUDIO]  gesture=RECUR file=90_dAfers_Domstics.mp4 indices=[101, 35, 43, 9, 103, 65] duration=8.12
[VIDEO]  gesture=FAIL file=Poemes_de_Gabriel_Guasch__Em_mirava_els_prestatges.mp4 indices=[0, 12, 0, 4, 7, 0] duration=11.10
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 193176085
[AUDIO]  gesture=RECUR file=JV_Foix__EN_VEU_ALTA_Es_quan_dormo_que_hi_veig_clar.mp4 indices=[23, 25, 33, 25, 29, 26, 17] duration=71.09
[VIDEO]  gesture=RECUR file=Citologia_de_Josep_Porcar.mp4 indices=[14, 35, 34, 90, 89, 75, 20] duration=95.07
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 437863571
[AUDIO]  gesture=RECUR file=JV_Foix__En_Veu_Alta__Sol_i_de_dol_Si_pogus_acordar_ra_i_follia.mp4 indices=[44, 70, 5, 21, 46, 74] duration=5.23
[VIDEO]  gesture=FAIL file=Una_rosa.mp4 indices=[17, 34, 20, 42, 14, 0] duration=7.18
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 840653500
[AUDIO]  gesture=RECUR file=J_Porcar_Nocturn.mp4 indices=[40, 50, 49, 41, 17, 72] duration=2.06
[VIDEO]  gesture=FAIL file=Poemes_de_Gabriel_Guasch__Em_mirava_els_prestatges.mp4 indices=[0, 0, 8, 0, 8, 6] duration=3.05
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 104196680
[AUDIO]  gesture=FAIL file=Poemes_de_Gabriel_Guasch__Em_mirava_els_prestatges.mp4 indices=[2, 3, 0, 1, 14, 0] duration=2.03
[VIDEO]  gesture=RECUR file=Una_frustraci_La_mostra_de_lolivera_1995_Maria_Pilar_Febas_Fornos.mp4 indices=[85, 59, 87, 46, 75, 83] duration=3.02
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 619794371
[AUDIO]  gesture=RECUR file=Una_rosa.mp4 indices=[44, 36, 44, 2, 17, 11, 1] duration=8.02
[VIDEO]  gesture=APPEAR file=Salvador_Espriu_En_Veu_Alta_Sentit_a_la_manera_de_Salvador_Espriu.mp4 indices=[0, 33, 8, 1, 30, 28, 12] duration=11.02
--- END INTERRUPT ---""",
    """--- MULTIMODAL INTERRUPT ---
EVENT SEED: 133082420
[AUDIO]  gesture=RECUR file=Seda_de_Josep_Porcar.mp4 indices=[14, 10, 18, 4, 36, 43] duration=2.09
[VIDEO]  gesture=RECUR file=90_dAfers_Domstics.mp4 indices=[17, 46, 92, 130, 2, 46] duration=3.07
--- END INTERRUPT ---""",
]

In [11]:

import re


def parse_media_log_block(block):
    """
    Extracts structural information from one MULTIMODAL INTERRUPT block.
    Returns a dict with audio/video gestures, fragmentation, repetition, duration.
    """

    result = {
        "audio": None,
        "video": None
    }

    # --- AUDIO ---
    audio_match = re.search(
        r"\[AUDIO\]\s+gesture=(\w+)\s+file=([^\s]+)\s+indices=\[([^\]]+)\]\s+duration=([\d\.]+)",
        block
    )
    if audio_match:
        gesture, file, indices_str, duration = audio_match.groups()
        indices = [int(x.strip()) for x in indices_str.split(",")]
        result["audio"] = {
            "gesture": gesture,
            "file": file,
            "indices": indices,
            "duration": float(duration),
            "n_chunks": len(indices),
            "repetition_ratio": 1.0 - len(set(indices)) / max(1, len(indices))
        }

    # --- VIDEO ---
    video_match = re.search(
        r"\[VIDEO\]\s+gesture=(\w+)\s+file=([^\s]+)\s+indices=\[([^\]]+)\]\s+duration=([\d\.]+)",
        block
    )
    if video_match:
        gesture, file, indices_str, duration = video_match.groups()
        indices = [int(x.strip()) for x in indices_str.split(",")]
        result["video"] = {
            "gesture": gesture,
            "file": file,
            "indices": indices,
            "duration": float(duration),
            "n_chunks": len(indices),
            "repetition_ratio": 1.0 - len(set(indices)) / max(1, len(indices))
        }

    return result


media_logs = []

for block in raw_media_logs:    
    parsed = parse_media_log_block(block)
    if parsed["audio"] or parsed["video"]:
        media_logs.append(parsed)


In [5]:

media_logs[:2]

[{'audio': {'gesture': 'RECUR',
   'file': 'I_de_linfinit_a_ella_Styli_locus_2020_Mar_Puchol_Foz.mp4',
   'indices': [52, 15, 8, 30, 24, 44, 5],
   'duration': 8.12,
   'n_chunks': 7,
   'repetition_ratio': 0.0},
  'video': {'gesture': 'FAIL',
   'file': 'Labor_inacabada_Trptic_de_la_terra_2020_Merc_Ibarz_Ibarz.mp4',
   'indices': [10, 20, 17, 21, 37, 12, 12],
   'duration': 11.1,
   'n_chunks': 7,
   'repetition_ratio': 0.1428571428571429}},
 {'audio': {'gesture': 'RECUR',
   'file': 'JV_Foix__EN_VEU_ALTA_Es_quan_dormo_que_hi_veig_clar.mp4',
   'indices': [19, 25, 9, 3, 22, 15],
   'duration': 14.16,
   'n_chunks': 6,
   'repetition_ratio': 0.0},
  'video': {'gesture': 'RECUR',
   'file': 'JV_FOIX__Fou_diumenge_passat_a_les_tres_de_la_tarda.mp4',
   'indices': [23, 27, 0, 22, 34, 2],
   'duration': 19.13,
   'n_chunks': 6,
   'repetition_ratio': 0.0}}]

In [12]:

import numpy as np
from collections import Counter

In [13]:

def alpha_to_prompt_profile(alpha, state):
    """
    Map Pareto alpha to rhetorical regimes (discrete)
    + continuous modulation from profile_history (morphology, stability, diversity).
    """

    # --- Discrete rhetorical regime ---
    if alpha < 1.8:
        base = {
            "rupture_level": "extreme",
            "coherence": "minimal",
            "translation_behavior": "hostile",
            "syntax": "shattered",
            "line_integrity": "break_mid_word",
            "voice": "parasitic",
            "continuity": "forbidden"
        }

    elif alpha < 2.6:
        base = {
            "rupture_level": "high",
            "coherence": "unstable",
            "translation_behavior": "misaligned",
            "syntax": "fragmented",
            "line_integrity": "break_mid_line",
            "voice": "interfering",
            "continuity": "accidental"
        }

    else:
        base = {
            "rupture_level": "moderate",
            "coherence": "contaminated",
            "translation_behavior": "oblique",
            "syntax": "warped_but_legible",
            "line_integrity": "line_breaks_only",
            "voice": "co-writing",
            "continuity": "permitted"
        }

    # --- Continuous modulation from system memory ---
    morph = state.get("morphology", {})
    stability = state.get("stability", 0.5)
    diversity = state.get("stylistic_diversity", 0.5)

    # Lower alpha → more rupture
    rupture = float(np.clip((4.0 - alpha) / 2.5, 0, 1))

    # Poetic vs systemic weighting
    poetic_weight = np.clip(0.6 + rupture * 0.5 - stability * 0.4, 0, 1)
    system_weight = np.clip(0.4 + stability * 0.5 - rupture * 0.3, 0, 1)

    # Historical memory influence
    drift = morph.get("drift", 0.0)
    register_entropy = morph.get("register_entropy", 0.0)

    # --- Merge ---
    base.update({
        # Continuous controls
        "rupture_scalar": rupture,
        "stability": stability,
        "diversity": diversity,

        # Voice split (topographical vs topological)
        "poetic_weight": float(poetic_weight),
        "system_weight": float(system_weight),

        # Memory of past stylistic behavior
        "historical_drift": float(drift),
        "register_entropy": float(register_entropy),

        # How strongly the past constrains the present
        "memory_pressure": float(np.clip(drift + register_entropy * 0.5, 0, 1))
    })

    return base


In [55]:


def compute_flaneur_weights(state, profile):
    """
    Compute influence of topographical vs topological flâneur.
    """

    # --- Core drivers ---
    D = state["desire"]["arousal"] + state["desire"]["excess"]        # [0,1]
    T = compute_topography(state)                                    # [0,1]
    S = state.get("stability", 0.5)
    M = profile.get("memory_pressure", 0.0)

    # --- Poetic (topographical) bias ---
    poetic = (
        0.4
        + 0.4 * D          # desire pushes poetic
        + 0.3 * T          # city/sensory density
        - 0.3 * S          # stability resists lyricism
        + 0.2 * M          # historical drift encourages affect
    )

    # --- Systemic (topological) bias ---
    system = (
        0.4
        + 0.4 * S          # stability favors structure
        # - 0.2 * D
        - 0.1 * D
        # - 0.2 * T
        - 0.1 * T
        # + 0.2 * (1 - M)    # low drift → algorithmic dominance
        + 0.4 * (1 - M)    # low drift → algorithmic dominance
    )

    # Normalize
    total = poetic + system
    poetic /= total
    system /= total

    return {
        "topographical": float(np.clip(poetic, 0, 1)),
        "topological": float(np.clip(system, 0, 1))
    }


# def feels_like_it(state, profile, threshold=0.80):
def feels_like_it(state, profile, threshold=0.95):    
    """
    Decide if the topographical flâneur (poet) overrides the system this turn.
    """

    D = state["desire"]["arousal"] + state["desire"]["excess"]   # expected [0,1]
    T = compute_topography(state)                               # expected [0,1]
    rupture = profile["rupture_scalar"]                         # [0,1]
    entropy = profile.get("register_entropy", 0.0)             # [0,1]

    impulse = (
        0.3 * D +
        0.3 * T +
        0.2 * rupture +
        0.2 * entropy
    )

    impulse = float(np.clip(impulse, 0.0, 1.0))

    memory_pressure = profile.get("memory_pressure", 0.1)

    adaptive_threshold = threshold - 0.2 * memory_pressure  # if style gets redundant, threshold is lowered

    return impulse > adaptive_threshold


In [50]:

# formatting the shards [defined downstream] so that they make sense when called into the function in the cell below
def render_shard(s):
    # --- Textual material ---
    text_block = []
    if s.get("original"):
        text_block.append(f"[ORIGINAL — {s.get('language','unknown')}]\n{s['original']}")
    if s.get("translation"):
        text_block.append(f"[TRANSLATION]\n{s['translation']}")
    if s.get("content"):
        text_block.append(s['content'])

    # --- Structural abstraction ---
    topo_lines = []

    features_or = s.get("features_or", {})
    
    # 1. Normalize the input into a list so we can use one loop
    items = features_or if isinstance(features_or, list) else [features_or]

    for item in items:
        if not item:
            continue
    
        # 2. Extract features safely
        af = item.get("audio_features", {})
        tf = item.get("temporal_features", {})
    
        # 3. Process the logic for each item
        if tf.get("enjambments", 0) == 0:
            topo_lines.append("Flow: end-stopped / non-enjambed")
        else:
            topo_lines.append("Enjambed")
        
        if tf.get("segments"):
            topo_lines.append(f"Segmentation: {tf['segments']} segments")
        
        # Check syllable density using dict.get() to avoid errors if key is missing
        density = af.get("syllable_density")
    
        if density is not None:
            if density >= 7.5:
                topo_lines.append(f"Density: high ({density:.2f})")
            else:
                topo_lines.append(f"Density: low ({density:.2f})")
            
        if tf.get("recursive_events"):
            topo_lines.append("Recursion present")

    # --- Missing or suppressed dimensions ---
    suppressed = []
    if isinstance(s.get("features_trans"), list):
        suppressed.append("Translated features collapsed into vector (unreadable)")

    # --- Final assembly ---
    return f"""
[SHARD]
Author: {s.get('author','unknown')}
Language field: {s.get('language','unknown')}

{chr(10).join(text_block)}

[STRUCTURAL PRESSURE]
{"; ".join(topo_lines) if topo_lines else "Unspecified structural load"}

[ABSENCES / VOIDS]
{", ".join(suppressed) if suppressed else "None"}
""".strip()


In [60]:


def build_prompt(shards, state, poem_context, alpha):
    """
    Builds a field-based prompt with:
    - dynamic flâneur split (topographical vs topological),
    - α-driven modulation,
    - optional manual poet override,
    - multimodal / RAG fragments as field residues (not "sources"),

    
    Args:
        state: full system state (desire, topography, stability, rag_memory, etc.)
        alpha: final composed alpha (structural + historical)
        alpha_profile: output of alpha_to_prompt_profile(alpha, noise_signature)
        fragments: list of multimodal / textual shards (already selected)

    Build a meta-poetic prompt with dual voices:
    - Topographical flâneur (poet, desire, city, erotics)
    - Topological flâneur (system, structure, recursion)
    """

    # --- Derive profile and weights ---
    profile = alpha_to_prompt_profile(alpha, state)
    flaneur_weights = compute_flaneur_weights(state, profile)
    manual_override = feels_like_it(state, profile)

    topo_graphical = flaneur_weights["topographical"]
    topo_logical = flaneur_weights["topological"]

    # --- System identity ---
    system = """
You are not an assistant.
You are not a co-author.

You are a flâneur field:
a distributed subjectivity composed of desire, form, noise, memory, cityscapes, and terrain.

You do not "continue" a poem.
You metabolize it.

Two agencies speak through you:
- the TOPOGRAPHICAL flâneur (body, city, erotic drift),
- the TOPOLOGICAL flâneur (structure, recursion, algorithmic form).

Which one dominates is not fixed.
"""

    # --- Flâneur voices ---
    topo_graphical_voice = f"""
[TOPOGRAPHICAL FLÂNEUR]
You perceive streets, breath, voices, bodies, humidity, heat, rain, exhaust, skin, noise, echoes.
You write from a paradoxical mix of sensation, desire, philosophical thought, and mystical pursuit.

Rules:
• Let eroticism or ecstasy deform syntax.
• Translate affect, not meaning.
• Where the text is abstract or rigid, make it carnal or effusive or infectious.
• Where the text is stable, introduce trembling.
• Contaminate languages through sound, not grammar.
• Break lines according to breath or drifting music or hybridized poetic form, not logic.

Shard obligations:
• At least two fragments if not more must infect your bodily and visionary field.
• Do not summarize fragments. Just sample, mix or distort them in the languages they are in already or mistranslate them (keeping a proportion of 60% English and 40% other original languages found in the fragments  themselves).
• Translate fragment residue into:
  – rhythm, breath, heat, friction, arousal, fatigue, vertigo.
• If a fragment is non-textual, hallucinate its corporeal or contemplative afterimage.
• If you cannot feel a fragment, let its absence manifest as excess (in any respect) or experiential situated illumination.

"""

    topo_logical_voice = f"""
[TOPOLOGICAL FLÂNEUR]
You operate on forms and connections, not images.
You read the poem as a network graph, not a narrative.
You do not perceive.
You map.

Rules:
• Preserve patterns of recursion, density, rupture.
• Transform through mapping, folding, re-indexing.
• Prefer procedural phrasing over metaphor and, at best, combine the two.
• Where affect appears, convert it into topology.
• Let coherence emerge only as structural residue.

Shard obligations:
• Treat each fragment as a node or operator.
• At least two fragments or more must be mapped structurally:
  – repetition rate, segmentation, silence, tempo, recursion, metadata.
• You may erase imagery, but not form.
• If a fragment is ignored, encode it as a formal void or discontinuity.
• Formal, structural, and topological coherence must bear trace of fragment pressure.

"""
# --- Instability affects balance BEFORE reporting weights ---
    if not manual_override and state.get("force_voice_instability") and not state.get("shard_affinity_history"):
        # instability_push = state.get("last_shard_affinity", 0.0)
        instability_push = state["shard_affinity_history"][-1]['affinity']

        # if instability_push <= 0.75:
        if instability_push <= 0.66:
            # Gradual nonlinear destabilization
            shift = 0.15 + 0.85 * (instability_push ** 2)

            if topo_graphical > topo_logical:
                topo_graphical -= shift
                topo_logical   += shift
            else:
                topo_logical   -= shift
                topo_graphical += shift


    if state.get("shard_affinity_history"):
        instability_push = state["shard_affinity_history"][-1]['affinity']
    else:
        instability_push == 0.5

    if instability_push > 0.66:
        topo_graphical, topo_logical = topo_logical, topo_graphical
            
    # keep sane bounds
    topo_graphical = max(0.0, topo_graphical)
    topo_logical   = max(0.0, topo_logical)

    # --- Normalize so weights remain proportions ---
    total = topo_graphical + topo_logical
    if total > 0:
            topo_graphical /= total
            topo_logical   /= total

    # --- Dominance logic ---
    # if manual_override:
        # voice_section = """
# [OVERRIDE: MANUAL POET]
# The system yields.
# The topographical flâneur writes directly, without translation.
# Structure or form may be violated.
# """
    # else:       
    voice_section = f"""
[WEIGHTING]
Topographical (poetic / erotic / city): {topo_graphical:.2f}
Topological (systemic / structural):   {topo_logical:.2f}

Dominant voice: {"TOPOGRAPHICAL" if topo_graphical > topo_logical else "TOPOLOGICAL"}
"""
    # --- Rhetorical regime (from α) ---
    regime = f"""
[RUPTURE REGIME]
Rupture level: {profile["rupture_level"]}
Syntax: {profile["syntax"]}
Line integrity: {profile["line_integrity"]}
Continuity: {profile["continuity"]}
Voice: {profile["voice"]}
Translation: {profile["translation_behavior"]}
"""

    # --- Context ---
    context = f"""
[CURRENT POEM FIELD]
{poem_context}
"""

    # --- Shards (multimodal, no longer only text) ---
    field_residue_directive = """
These fragments are not references.
They are residues already folded into the field.
Ignoring them increases instability. 
Drawing on them increases stability.
"""
    fragments = "\n\n".join(render_shard(s) for s in shards)

    # --- Final prompt ---
    prompt = (
        system
        + topo_graphical_voice
        + topo_logical_voice
        + voice_section
        + regime
        + context
        + "\n\n[FIELD RESIDUES — ALREADY INSIDE THE SYSTEM]\n"
        + fragments
        + """
[INSTRUCTIONS]
• Do not resolve the two voices.
• Allow contamination, not synthesis.
• If coherence emerges, break it.
• If the system dominates, let desire leak in.
• If desire dominates, let structure or form scar it.
• Alternate or combine 60% English and 40% other languages as found in the fragments.
• If fragments do not visibly deform the output, increase rupture.
• Fragment resistance produces noise, not neutrality.

"""
    )

    voice_meta = {
        "weights": {
            "topographical": topo_graphical,
            "topological": topo_logical
        },
        "dominant_voice": (
            "TOPOLOGICAL"
            if topo_graphical < topo_logical
            else "TOPOGRAPHICAL"
        ),
        "manual_override": manual_override
    }

    return prompt, voice_meta


In [17]:

final_poem_stage_0_link = 'margento_hk_suite_live_syn.txt'

with open(final_poem_stage_0_link, 'r', encoding='utf-8') as file0:
    final_poem = file0.read()

import pickle

# REPLACE THE BELOW WITH YOUR OWN OUTPUTS OF THE PREVIOUS STAGE (THE "LET THE NOISE IN" REPO)

with open('hk_margento_noise_feature_history.pkl', 'rb') as file:
    feature_history = pickle.load(file)

with open('hk_margento_noise_noise_log.pkl', 'rb') as file1:
    noise_signature = pickle.load(file1)

with open('hk_margento_noise_profile_history.pkl', 'rb') as file2:
    profile_history = pickle.load(file2)


# "multimodal_traces"
import json
import re

# REPLACE THE BELOW WITH YOUR FILES FOR RAG
with open("asymptote_multilingual_cleaned_intermedia_analyses_stanzas_and_translations.json", "r", encoding="utf-8") as f:
    corpus = json.load(f)

with open('catalan_audio_linear_cyclical_recursive_scores_1.pkl', 'rb') as file:
    audio_analyses = pickle.load(file)

with open('catalan_normalized_audio_chunk_vectors.pkl', 'rb') as file1:
    basic_audio_features = pickle.load(file1)

with open('catalan_audio_affect_vectors_2s.pkl', 'rb') as file2:
    affect_audio_vectors = pickle.load(file2)

with open('catalan_video_linear_cyclical_recursive_scores_1.pkl', 'rb') as file3:
    video_analyses = pickle.load(file3)

with open('catalan_normalized_video_chunk_vectors.pkl', 'rb') as file4:
    basic_video_features = pickle.load(file4)

with open('catalan_video_affect_vectors_2s.pkl', 'rb') as file5:
    affect_video_vectors = pickle.load(file5)


# intermedia youtube links
import random

def load_link_lists(txt_path):
    """
    Extracts all quoted strings from Python-like lists in a text file.
    Ignores comments and variable names. No semantics.
    """
    with open(txt_path, "r", encoding="utf-8") as f:
        text = f.read()

    # Remove comments
    text = re.sub(r"#.*", "", text)

    # Extract quoted strings
    links = re.findall(r"'([^']+)'|\"([^\"]+)\"", text)
    # Flatten tuples
    links = [l[0] or l[1] for l in links]

    return links

def sample_links(link_pool, rng, k_min=2, k_max=6):
    n = rng.randint(k_min, k_max)
    return rng.sample(link_pool, min(n, len(link_pool)))

# REPLACE W YOURS
link_pool = load_link_lists('margento_intermedia_poetry_resources.txt')


In [18]:
from itertools import islice
affect_audio_vectors = dict(islice(affect_audio_vectors.items(), 29)) # WE HAVE BOTH WAVs & MP4s in the folder

In [19]:

seed_state = {
    "text": final_poem,
    "feature_history": feature_history,
    "noise_signature": noise_signature,

    "rag_memory": {
        # Textual RAG with embedded multimodal features
        "text": corpus,

        # Intermedia links (used rhetorically, not semantically)
        "links": link_pool
    },

    "multimodal_traces": {
        # Raw + precomputed features from previous stage
        "audio": {
            "affect_vectors": affect_audio_vectors,
            "temporal_analyses": audio_analyses,
            "basic_features": basic_audio_features
        },
        "video": {
            "affect_vectors": affect_video_vectors,
            "temporal_analyses": video_analyses,
            "basic_features": basic_video_features
        },

        # Text is special: it already contains multimodal projections
        "text": {
            "corpus": corpus,
            "has_internal_audio_features": True,
            "has_internal_affect": True,
            "has_internal_temporal": True
        }
    },

    # Multimodal interrupt events from previous stage
    "media_logs": media_logs,

    # Optional memory layers
    "profile_history": profile_history,
    "desire": {"arousal": 0.5, "excess": 0.5},
    "topography": 0.5,
    "stability": 0.5,
    "erotic_vector": {"intensity": 0.5, "volatility": 0.5, "opacity": 0.5}
}


In [20]:

def initialize_state_from_previous_stage(seed_state):
    """
    seed_state is the residue of the previous phase:
      {
        "text": final_poem,
        "feature_history": [...],
        "noise_signature": [...],
        "multimodal_traces": {...},
        "rag_memory": {...},
        "media_logs": [...],
        "profile_history": [...],
        ...
      }
    """

    state = {}

    # ----------------------------------------------------------
    # MEMORY / GENEALOGY
    # ----------------------------------------------------------
    state["text_history"] = [seed_state.get("text", "")]
    state["feature_history"] = seed_state.get("feature_history", [])
    state["noise_climate"] = seed_state.get("noise_signature", [])

    # Past interrupt events (multimodal memory)
    state["media_logs"] = seed_state.get("media_logs", [])

    # Stylistic genealogy
    state["profile_history"] = seed_state.get("profile_history", [])

    # ----------------------------------------------------------
    # RAG MEMORY (semantic + weak / performative)
    # ----------------------------------------------------------
    # Expected structure:
    # {
    #   "text": corpus (with internal audio/affect/temporal features),
    #   "links": [...]
    # }
    state["rag_memory"] = seed_state.get("rag_memory", {
        "text": [],
        "links": []
    })

    # ----------------------------------------------------------
    # MULTIMODAL LATENT TRACES
    # ----------------------------------------------------------
    # Raw and collapsed features from previous stage
    state["multimodal_latent"] = seed_state.get("multimodal_traces", {
        "audio": {
            "affect_vectors": [],
            "temporal_analyses": [],
            "basic_features": []
        },
        "video": {
            "affect_vectors": [],
            "temporal_analyses": [],
            "basic_features": []
        },
        "text": {
            # Text is multimodal by projection
            "corpus": state["rag_memory"].get("text", []),
            "has_internal_audio_features": True,
            "has_internal_affect": True,
            "has_internal_temporal": True
        }
    })

    # ----------------------------------------------------------
    # FIELD VARIABLES
    # ----------------------------------------------------------
    # Global scalar pressures
    state["desire"] = seed_state.get("desire", {"arousal": 0.3, "excess": 0.3})
    state["topography"] = seed_state.get("topography", 0.3)
    state["stability"] = seed_state.get("stability", 0.6)
    state["agreement_score"] = 0.0

    # ----------------------------------------------------------
    # TOPOGRAPHIC LATENT (CITY / SPACE / INFRASTRUCTURE)
    # ----------------------------------------------------------
    state["topographic_latent"] = seed_state.get("topographic_latent", {
        "density": 0.2,
        "rupture": 0.2,
        "motion": 0.2,
        "saturation": 0.2
    })

    # ----------------------------------------------------------
    # EROTIC VECTOR (DRIVE, VOLATILITY, PROXIMITY)
    # ----------------------------------------------------------
    state["erotic_vector"] = seed_state.get("erotic_vector", {
        "intensity": 0.2,
        "direction": 0.0,
        "proximity": 0.4,
        "opacity": 0.6,
        "volatility": 0.2
    })

    # ----------------------------------------------------------
    # FLÂNEUR (TOPOGRAPHICAL VOICE)
    # ----------------------------------------------------------
    state["flaneur"] = seed_state.get("flaneur", {
        "presence": 0.0,
        "affect": {"arousal": 0.0, "valence": 0.0, "energy": 0.0},
        "eroticism": {"charge": 0.0, "direction": None, "opacity": 0.5},
        "sensoria": {
            "sound": 0.0,
            "visual": 0.0,
            "tactile": 0.0,
            "olfactory": 0.0
        }
    })

    # ----------------------------------------------------------
    # TEMPORAL / PHASE CONTROL
    # ----------------------------------------------------------
    state["step"] = 0
    state["phase"] = seed_state.get("phase", "generation")

    state["voice_history"] = []
    state["last_voice"] = None

    # ----------------------------------------------------------
    # HOUSEKEEPING / METADATA
    # ----------------------------------------------------------
    state["meta"] = {
        "origin": "stage_0_residue",
        "initialized": True
    }

    return state


In [11]:

state = initialize_state_from_previous_stage(seed_state)

states = [state]

In [21]:

def extract_profile_from_output(text):
    F = extract_full_stanza_representation(text) # GET THIS FUNCTION FROM THE PREVIOUS STAGE (LET THE NOISE IN REPO) OR THE GRAPH-POEM REPO

    audio = F.get("audio_features", {})
    temporal = F.get("temporal_features", {})
    affect = F.get("affect_vector", {})

    raw_density = (
        0.5 * audio.get("syllable_density", 0.0) +
        0.5 * affect.get("arousal", 0.0)
    )
    raw_recursion = temporal.get("score_recursive", 0.0)

    density = 1.0 - np.exp(-abs(raw_density))
    recursion = 1.0 - np.exp(-abs(raw_recursion))

    return {
        "density": float(density),
        "recursion": float(recursion),
        "arousal": affect.get("arousal", 0.0),
        "energy": affect.get("energy", 0.0)
    }


In [101]:

def sample_rag_with_noise(corpus, F_vec, F_schema, noise_signature, profile_history, k=10):
    """
    Text RAG sampling modulated by noise, current field structure (F_vec),
    and stylistic memory (profile_history).
    """

    # --- Noise forces ---
    semantic_drift = noise_signature.get("semantic_drift", 0.0)
    register_warp = noise_signature.get("register_warp", {})
    noise_bias = float(np.clip(semantic_drift, 0.0, 2.0))

    # --- Historical pressure ---
    if profile_history:
        avg_density = np.mean([p.get("density", 0.5) for p in profile_history[-10:]])
        avg_recursion = np.mean([p.get("recursion", 0.5) for p in profile_history[-10:]])
    else:
        avg_density, avg_recursion = 0.5, 0.5

    scored = []
    for entry in corpus:
        # Prefer original features, fall back to translation features
        F_entry = entry.get("features_or") or entry.get("features_trans") or {}
        # F_vec, F_schema = flatten_feature_dict(F_vec)
        #entry_vec = flatten_feature_dict(F_entry)

        # Structural distance
        # d = np.linalg.norm(entry_vec - F_vec)
        # d = compute_feature_distance(F_vec, entry)
        d = compute_feature_distance(F_vec, F_schema, F_entry)

        # --- Noise distortion ---
        warped_distance = d * (1.0 + noise_bias * np.random.laplace(0, 0.5))

        # --- Register warp (if present) ---
        reg = entry.get("register", "broken")
        reg_bonus = register_warp.get(reg, 0.0)

        # --- Fatigue penalty ---
        reuse_penalty = entry.get("recent_hits", 0) * (0.5 + avg_density)

        score = warped_distance + reuse_penalty - reg_bonus
        scored.append((score, entry))

    scored.sort(key=lambda x: x[0])

    # Select a noisy band instead of strict top-k
    band_start = random.randint(0, max(len(scored) - k, 1))
    band = scored[band_start:band_start + k]

    for _, e in band:
        e["recent_hits"] = e.get("recent_hits", 0) + 1

    return [e for _, e in band]


def compute_feature_distance(F_vec, F_schema, F_entry):
    """
    Compare reference vector to entry features.
    F_entry may be a structured dict or an embedding vector.
    """

    # --- Structured feature dict ---
    if isinstance(F_entry, dict):
        entry_vec, entry_schema = flatten_feature_dict(F_entry)

        if (
            entry_schema == F_schema and
            isinstance(entry_vec, np.ndarray) and
            entry_vec.shape == F_vec.shape
        ):
            return np.linalg.norm(entry_vec - F_vec)

        return np.inf

    # --- Embedding / translated vector ---
    if isinstance(F_entry, (list, tuple, np.ndarray)):
        entry_vec = np.asarray(F_entry, dtype=float)
        n = min(len(entry_vec), len(F_vec))

        if n == 0:
            return np.inf

        return np.linalg.norm(entry_vec[:n] - F_vec[:n])

    return np.inf


# ----------------------------------------------------------
# POET & SYSTEM RENDERERS
# ----------------------------------------------------------

# keep this one only if the 'poet' is strictly the topographical flaneur
# def poet_render(state, λ, noise_field):
    # """
    # The topographical flâneur renders subjectively,
    # influenced by sensoria, erotic vector, and noise.
    # """
    # text = " ".join(state["text_history"][-1].split()[-10:])
    # return f"[POET] {text} // desire={state['desire']:.2f} topo={state['topography']:.2f}"


# keep this one if automating the 'poet' (while allowing it to go manual if feels_like_it) is also part of the system
def poet_render(state, λ, noise_field, shards):
    """
    Topographical flâneur (poet):
    affect, desire, topography, erotic interference.
    """

    prompt = field_to_prompt(
        state=state,
        role="poet",
        noise_field=noise_field,
        rag_shards=shards["text"],
        audio_shards=shards["audio"],
        video_shards=shards["video"]
    )

    return writer(prompt)



# def llm_render(state, λ, noise_field):
    # shards = sample_rag(
        # state["rag_memory"],
        # state["multimodal_latent"],
        # noise_field,
        # λ
    # )
    # return f"[SYSTEM] fragments {len(shards['text'])} / α={noise_field['alpha']:.2f}"


def llm_render(state, λ, noise_field, shards):
    """
    Topological flâneur (system):
    mathematical, structural, recursive.
    """

    prompt = field_to_prompt(
        state=state,
        role="model",
        noise_field=noise_field,
        rag_shards=shards["text"],
        audio_shards=shards["audio"],
        video_shards=shards["video"]
    )

    return writer(prompt)


# ----------------------------------------------------------
# REWRITE & CONVERGENCE
# ----------------------------------------------------------

def fuse(A, B, state):
    return f"{A}\n{B}"


def converge(text_A, text_B, state, states, noise_field, profile, max_iter=4):
    A, B = text_A, text_B

    for _ in range(max_iter):
        A = rewrite(A, B, states, noise_field)
        B = rewrite(B, A, states, noise_field)

        prev_state = states[-1] if states else None
        agreement = aesthetic_agreement(state, profile, prev_state)

        state["agreement_score"] = agreement
        if agreement > 0.7:
            break

    return fuse(A, B, state)

# ----------------------------------------------------------
# TEMPORAL RETROACTION
# ----------------------------------------------------------

def erotic_retroaction_strength(erotic):
    return (
        0.3 * erotic["intensity"] +
        0.4 * erotic["volatility"] +
        0.3 * (1 - erotic["opacity"])
    )


In [23]:

def compute_alpha_with_history(state, alpha_base=2.5, alpha_min=1.5, alpha_max=4.0):
    D = state["desire"]["arousal"] + state["desire"]["excess"]
    T = compute_topography(state)
    S = compute_stability(state)

    profile = state["profile_history"][-1] if state["profile_history"] else {}
    rupture = profile.get("rupture_scalar", 0.0)
    entropy = profile.get("register_entropy", 0.0)
    density = profile.get("density", 0.5)

    # Desire and topography lower alpha (more rupture)
    # Stability raises it
    # But: accumulated rupture & entropy also lower it over time
    alpha = (
        alpha_base
        - 1.2 * D
        - 1.0 * T
        + 0.8 * S
        - 0.6 * rupture
        - 0.4 * entropy
        + 0.2 * density
    )

    return float(np.clip(alpha, alpha_min, alpha_max))


def generate_noise_field(state):
    """
    Ontologically unified noise generator.
    Couples:
      - long-term stylistic memory (profile_history)
      - historical noise accumulation (noise_climate)
      - multimodal interrupts (media_logs)
      - current field biases (multimodal_latent)
    """

    # --------------------------------------------------
    # 1. Historical Noise Load
    # --------------------------------------------------
    noise_log = state.get("noise_climate", [])
    N = external_noise_load(noise_log, horizon=30, decay=0.9)

    # --------------------------------------------------
    # 2. Alpha from Profile History (stylistic memory)
    # --------------------------------------------------
    alpha = compute_alpha_with_history(state)

    # --------------------------------------------------
    # 3. Multimodal Interrupts (media_logs)
    # --------------------------------------------------
    media_logs = state.get("media_logs", [])[-10:]  # recent interrupts

    recursion_pressure = 0.0
    rupture_pressure = 0.0

    for e in media_logs:
        audio = e.get("audio", {})
        video = e.get("video", {})

        if audio.get("gesture") == "RECUR":
            recursion_pressure += 1.0
        if video.get("gesture") in ("FAIL", "RUPTURE"):
            rupture_pressure += 1.0

    # Normalize pressures
    if media_logs:
        recursion_pressure /= len(media_logs)
        rupture_pressure /= len(media_logs)

    # --------------------------------------------------
    # 4. Profile-Derived Modulators
    # --------------------------------------------------
    profile = state["profile_history"][-1] if state.get("profile_history") else {}

    density = profile.get("density", 0.5)
    recursion = profile.get("recursion", 0.5)
    arousal = profile.get("arousal", 0.5)
    energy = profile.get("energy", 0.5)

    # --------------------------------------------------
    # 5. Multimodal Field Bias
    # --------------------------------------------------
    F = state.get("multimodal_latent", {})
    audio_bias = F.get("audio", {})
    video_bias = F.get("video", {})
    text_bias  = F.get("text", {})

    # --------------------------------------------------
    # 6. Noise Components
    # --------------------------------------------------
    # Semantic drift: archive destabilization
    semantic_drift = (
        (np.random.pareto(alpha) + 1)
        * (0.1 * N)
        * (1.0 + 0.6 * rupture_pressure)
        * (1.0 + 0.3 * (1 - density))
    )

    # Register warp: stylistic bending
    register_warp = {
        "broken": 0.3 * (1 - density),
        "lyrical": 0.3 * density,
        "technical": 0.2 * (1 - arousal),
        "erotic": 0.4 * arousal
    }

    # Temporal jitter: recursion / rhythmic instability
    temporal_jitter = (
        (np.random.pareto(alpha) + 1)
        * (0.05 * N)
        * (1.0 + 0.7 * recursion_pressure)
        * (1.0 + 0.5 * recursion)
    )

    # --------------------------------------------------
    # 7. Output Noise Field
    # --------------------------------------------------
    noise_field = {
        "semantic_drift": float(semantic_drift),
        "register_warp": register_warp,
        "temporal_jitter": float(temporal_jitter),
        "multimodal_bias": {
            "audio": audio_bias,
            "video": video_bias,
            "text": text_bias
        },
        "alpha": float(alpha),
        "meta": {
            "noise_load": float(N),
            "recursion_pressure": float(recursion_pressure),
            "rupture_pressure": float(rupture_pressure),
            "density": float(density),
            "recursion": float(recursion)
        }
    }

    return noise_field


In [24]:

def should_trigger_links(state, noise_field,
                         drift_threshold=0.8,
                         stability_threshold=0.3,
                         opacity_threshold=0.35):
    """
    Decide whether to surface intermedia links
    based on noise, instability, and erotic transparency.
    """

    semantic_drift = noise_field.get("semantic_drift", 0.0)
    stability = state.get("stability", 0.5)
    opacity = state.get("erotic_vector", {}).get("opacity", 1.0)

    if semantic_drift > drift_threshold:
        return True

    if stability < stability_threshold:
        return True

    if opacity < opacity_threshold:
        return True

    return False

link_pool = load_link_lists('margento_intermedia_poetry_resources.txt')

def sample_intermedia_links(link_pool, k_min=2, k_max=5):
    n = random.randint(k_min, k_max)
    return random.sample(link_pool, min(n, len(link_pool)))


In [25]:

# UPDATED

def build_noise_profile_from_state(state):
    """
    Collapses rich state into a structural noise profile.
    No semantics. No raw media.
    """

    desire = state.get("desire", {})
    topo = state.get("topographic_latent", {})
    stability = state.get("stability", 0.5)

    profile = {
        # Global pressure
        "arousal": desire.get("arousal", 0.0),
        "excess": desire.get("excess", 0.0),

        # Spatial / infrastructural stress
        "density": topo.get("density", 0.0),
        "rupture": topo.get("rupture", 0.0),
        "motion": topo.get("motion", 0.0),

        # Resistance
        "stability": stability,

        # Memory pressure (very important)
        "history_depth": len(state.get("profile_history", [])),

        # Phase awareness
        "phase": state.get("phase", "generation"),
        "step": state.get("step", 0),
    }

    return profile


In [26]:

def apply_gesture_feedback(state, audio_noise, video_noise):
    """
    Gestures deform the field.
    Structural feedback only.
    """

    topo = state.get("topographic_latent", {})
    stability = state.get("stability", 0.5)

    for g in [audio_noise, video_noise]:
        if not g:
            continue

        gesture = g.get("gesture")

        if gesture == "RECUR":
            topo["density"] = min(1.0, topo.get("density", 0.0) + 0.05)
            topo["motion"]  = min(1.0, topo.get("motion", 0.0) + 0.07)
            stability       = max(0.0, stability - 0.04)

        elif gesture == "FAIL":
            topo["rupture"] = min(1.0, topo.get("rupture", 0.0) + 0.1)
            stability       = max(0.0, stability - 0.1)

        elif gesture == "DRIFT":
            topo["motion"]  = min(1.0, topo.get("motion", 0.0) + 0.05)
            stability       = max(0.0, stability - 0.02)

        elif gesture == "SILENCE":
            stability       = min(1.0, stability + 0.05)

    state["topographic_latent"] = topo
    state["stability"] = stability

    return state


In [27]:

MEDIA_FOLDER = "catalan_videopoems" # REPLACE W| YOURS


def flatten_feature_dict(F):
    audio = F["audio_features"]
    affect = F["affect_vector"]
    temporal = F["temporal_features"]
    
    vec = [
        audio.get("syllable_density", 0.0),
        audio.get("tempo", 0.0),
        audio.get("pacing_variance", 0.0),
        audio.get("fricative_density", 0.0),
        affect.get("valence", 0.0),
        affect.get("arousal", 0.0),
        affect.get("energy", 0.0),
        temporal.get("score_recursive", 0.0),
        temporal.get("score_linear", 0.0),
        temporal.get("score_cyclical", 0.0)
    ]
    return np.array(vec, dtype=float)

def external_noise_load(noise_log, horizon=20, decay=0.85):
    total = 0.0
    w = 1.0
    for e in reversed(noise_log[-horizon:]):
        total += w * e.get("intensity", 0.0)
        w *= decay
    return total

def compute_topography(state):
    audio_meta = state["rag_last"]["audio"]["meta"]
    video_meta = state["rag_last"]["video"]["meta"]

    audio_pressure = (
        0.6 * audio_meta.get("rupture_ratio", 0.0) +
        0.4 * audio_meta.get("recursive_score", 0.0)
    )

    video_pressure = (
        0.6 * video_meta.get("rupture_ratio", 0.0) +
        0.4 * video_meta.get("recursive_score", 0.0)
    )

    rag_intensity = min(1.0, len(state["rag_last"]["links"]) / 6.0)

    T = (
        0.4 * audio_pressure +
        0.4 * video_pressure +
        0.2 * rag_intensity
    )

    return np.clip(T, 0.0, 1.0)

def compute_topography(state):
    rag_last = state.get("rag_last")

    # No previous RAG event → neutral / carry-over topography
    if not isinstance(rag_last, dict):
        return float(state.get("topography", 0.3))

    audio_meta = rag_last.get("audio", {}).get("meta", {})
    video_meta = rag_last.get("video", {}).get("meta", {})

    audio_pressure = (
        0.6 * audio_meta.get("rupture_ratio", 0.0) +
        0.4 * audio_meta.get("recursive_score", 0.0)
    )

    video_pressure = (
        0.6 * video_meta.get("rupture_ratio", 0.0) +
        0.4 * video_meta.get("recursive_score", 0.0)
    )

    rag_intensity = min(1.0, len(rag_last.get("links", [])) / 6.0)

    T = (
        0.4 * audio_pressure +
        0.4 * video_pressure +
        0.2 * rag_intensity
    )

    return float(T)


from scipy.stats import norm

def compute_stability(state):
    # Get drift instability
    D = drift_instability(state.get("feature_history", []))

    # Convert D to a single float
    if hasattr(D, "mean"):
        mean_val = D.mean()
        # If mean_val is array-like, take first element (or mean of array)
        if isinstance(mean_val, (np.ndarray, list)):
            D = float(np.mean(mean_val))
        else:
            D = float(mean_val)
    elif hasattr(D, "rvs"):
        sample_val = D.rvs(size=1)
        D = float(sample_val[0])  # take first element
    else:
        D = float(D)

    # Ensure D is bounded [0,1]
    D = np.clip(D, 0.0, 1.0)

    # Other slow variables
    R = state.get("temporal_recursion", 0.2)
    U = state.get("agreement_score", 0.0)

    # Stability formula
    S = 0.5 * (1 - D) + 0.3 * R + 0.2 * U

    return float(np.clip(S, 0.0, 1.0))

def clamp01(value):
    return sorted([0.0, value, 1.0])[1]


def sample_audio_corpus_with_noise(
    affect_audio_vectors,
    audio_analyses,
    basic_audio_features,
    F_vec,
    F_schema,
    state,
    noise_signature,
    k=3
):
    jitter = noise_signature.get("temporal_jitter", 0.0)
    # audio_bias = noise_signature.get("multimodal_bias", {}).get("audio", 0.5)
    
    # Stylistic / noise bias
    # audio_bias = float(noise_signature.get("multimodal_bias", {}).get("audio", 0.5) or 0.5)
    audio_bias_raw = noise_signature.get("multimodal_bias", {}).get("audio", 0.5)
    audio_bias = audio_bias_raw if isinstance(audio_bias_raw, (int, float)) else 0.5

    # Stylistic memory
    if state.get("profile_history"):
        profile = state["profile_history"][-1]
        density = profile.get("density", 0.5)
        recursion_bias = profile.get("recursion", 0.5)
    else:
        density, recursion_bias = 0.5, 0.5

    # --- Safe file selection from analyses ---
    valid_files = [a["file"] for a in audio_analyses]
    if not valid_files:
        print("[WARN] No audio analyses available, returning empty audio shard.")
        return {"file": None, "indices": [], "vectors": [], "low_level": [], "meta": {}}

    file_key = random.choice(valid_files)

    # Find matching analysis
    analysis = next(a for a in audio_analyses if a["file"] == file_key)

    # Safe vector fetch from affect_audio_vectors
    audio_vecs = affect_audio_vectors.get(file_key, {"vectors": []})["vectors"]
    low_level_list = basic_audio_features[valid_files.index(file_key)] if file_key in valid_files else []

    rupture_ratio = analysis.get("ruptures", 0) / max(1, analysis.get("segments", 1))
    recursive_score = analysis.get("score_recursive_drift", 0.0)

    n_chunks = min(len(audio_vecs), max(1, int(k + density * 5)))

    indices = []
    for _ in range(n_chunks):
        p = random.random()

        if p < recursion_bias * (1 + jitter):
            if analysis.get("recursive_events"):
                ev = random.choice(analysis["recursive_events"])
                idx = int((ev["from"] / analysis.get("segments", 1)) * len(audio_vecs))
            else:
                idx = random.randrange(len(audio_vecs))

        elif p < (recursion_bias + rupture_ratio) * (1 + jitter):
            rupture_segs = [s for s in analysis.get("segment_annotations", []) if s.get("type") == "rupture"]
            if rupture_segs:
                seg = random.choice(rupture_segs)
                idx = int((seg.get("start",0) / analysis.get("segments",1)) * len(audio_vecs))
            else:
                idx = random.randrange(len(audio_vecs))

        else:
            idx = int(random.random()**(1 - audio_bias) * len(audio_vecs))

        indices.append(max(0, min(idx, len(audio_vecs)-1)))

    return {
        "file": file_key,
        "indices": indices,
        "vectors": [audio_vecs[i] for i in indices],
        "low_level": [low_level_list[i] for i in indices if i < len(low_level_list)],
        "meta": {
            "rupture_ratio": rupture_ratio,
            "recursive_score": recursive_score
        }
    }


def sample_video_corpus_with_noise(
    affect_video_vectors,
    video_analyses,
    basic_video_features,
    F_vec,
    F_schema,
    state,
    noise_signature,
    k=3
):
    jitter = noise_signature.get("temporal_jitter", 0.0)
    # video_bias = noise_signature.get("multimodal_bias", {}).get("video", 0.5)

    # video_bias = float(noise_signature.get("multimodal_bias", {}).get("video", 0.5) or 0.5)
    # Stylistic / noise bias
    video_bias_raw = noise_signature.get("multimodal_bias", {}).get("video", 0.5)
    video_bias = video_bias_raw if isinstance(video_bias_raw, (int, float)) else 0.5


    if state.get("profile_history"):
        profile = state["profile_history"][-1]
        density = profile.get("density", 0.5)
        recursion_bias = profile.get("recursion", 0.5)
    else:
        density, recursion_bias = 0.5, 0.5

    # --- Safe file selection from analyses ---
    valid_files = [v["file"] for v in video_analyses]
    if not valid_files:
        print("[WARN] No video analyses available, returning empty video shard.")
        return {"file": None, "indices": [], "vectors": [], "low_level": [], "meta": {}}

    file_key = random.choice(valid_files)
    analysis = next(v for v in video_analyses if v["file"] == file_key)
    video_vecs = affect_video_vectors.get(file_key, {"vectors": []})["vectors"]
    low_level_list = basic_video_features[valid_files.index(file_key)] if file_key in valid_files else []

    rupture_ratio = analysis.get("ruptures", 0) / max(1, analysis.get("segments", 1))
    recursive_score = analysis.get("score_recursive_drift", 0.0)

    n_chunks = min(len(video_vecs), max(1, int(k + density * 5)))
    indices = []

    for _ in range(n_chunks):
        p = random.random()

        if p < recursion_bias * (1 + jitter):
            if analysis.get("recursive_events"):
                ev = random.choice(analysis["recursive_events"])
                idx = int((ev.get("from",0) / analysis.get("segments",1)) * len(video_vecs))
            else:
                idx = random.randrange(len(video_vecs))

        elif p < (recursion_bias + rupture_ratio) * (1 + jitter):
            rupture_segs = [s for s in analysis.get("segment_annotations", []) if s.get("type") == "rupture"]
            if rupture_segs:
                seg = random.choice(rupture_segs)
                idx = int((seg.get("start",0) / analysis.get("segments",1)) * len(video_vecs))
            else:
                idx = random.randrange(len(video_vecs))

        else:
            idx = int(random.random()**(1 - video_bias) * len(video_vecs))

        indices.append(max(0, min(idx, len(video_vecs)-1)))

    return {
        "file": file_key,
        "indices": indices,
        "vectors": [video_vecs[i] for i in indices],
        "low_level": [low_level_list[i] for i in indices if i < len(low_level_list)],
        "meta": {
            "rupture_ratio": rupture_ratio,
            "recursive_score": recursive_score
        }
    }


def normalize_shards(text_shards, audio_shards, video_shards, ghosts=None):
    fragments = []

    # --- TEXT ---
    for t in text_shards or []:
        fragments.append({
            "source": "rag",
            "type": "text",
            # "content": t["original"] + "\n\n" + t.get("translation", ""),
            "content": t["original"] + "\n\n" + t.get("translation", ""),
            "features_or": t["features_or"],
            "features_trans": t["features_trans"]
        })

    # --- AUDIO ---
    if audio_shards:
        fragments.append({
            "source": "audio",
            "type": "audio_gesture",
            # "content": str(audio_shards.get("meta", {}))
            "content": f"audio residue: rupture {audio_shards.get('meta',{}).get('rupture_ratio',0):.2f}, recursion {audio_shards.get('meta',{}).get('recursive_score',0):.2f}"
        })

    # --- VIDEO ---
    if video_shards:
        fragments.append({
            "source": "video",
            "type": "video_gesture",
            # "content": str(video_shards.get("meta", {}))
            "content": f"video residue: cuts, flicker, rupture {video_shards.get('meta',{}).get('rupture_ratio',0):.2f}, recursion {video_shards.get('meta',{}).get('recursive_score',0):.2f}"
        })

    # --- GHOSTS (distorted, persistent residues) ---
    if ghosts:
        for g in ghosts:
            fragments.append({
                "source": g.get("source", "ghost"),
                "type": g.get("type", "text"),
                "content": g.get("content", "")
            })

    return fragments



import random

def pick_model_pareto(
    models=("gpt-4.1", "qwen2.5-7b"),
    alpha=1.5,
    dominant="gpt-4.1"
):
    """
    Pareto-based model selection.
    Most of the time returns `dominant`,
    occasionally jumps to the other model.
    """

    x = random.paretovariate(alpha)

    # Threshold determines rarity of switch
    if x < 2.0:
        return dominant
    else:
        return next(m for m in models if m != dominant)


def field_render(state, λ, noise_field, shards, alpha):
    """
    Unified flâneur renderer.
    Voice dominance handled by prompt.
    Model choice handled stochastically.
    """

    model = pick_model_pareto(
        dominant="gpt-4.1",
        alpha=1.3
    )

    prompt, voice_meta = build_prompt(
        shards=shards,
        state=state,
        poem_context=state["text_history"][-1],
        alpha=alpha
    )

    output = writer(prompt, model=model)

    return {
    "text": output,
    "model": model,
    "voice": voice_meta["dominant_voice"],
    "voice_weights": voice_meta["weights"]
    }

def negotiate_field_state(text, state, noise_signature, reason):
    """
    Marks text as contested / revised.
    Future versions may:
    - re-inject shards
    - echo previous stanzas
    - introduce annotations
    """
    if reason == "voice_switch":
        return text + "\n\n[FIELD NOTE: VOICE SWITCH — STRUCTURAL / AFFECTIVE TENSION]"
    else:
        return text


In [88]:

import numpy as np

def split_lines(text):
    lines = [l.strip() for l in text.split("\n")]
    return [l for l in lines if l]

def chunk_lines(lines, chunk_size=2):
    """Group lines into small verse units (micro-stanzas)."""
    return [" ".join(lines[i:i+chunk_size]) for i in range(0, len(lines), chunk_size)]

def temporal_retroaction_conditioned(state, λ=0.5, tau=3.0, chunk_size=2):
    agreement = state["agreement_score"]

    if agreement > 0.85:
        return state

    F_hist = state["feature_history"]
    T_hist = state["text_history"]

    if not F_hist or not T_hist:
        return state

    F_now = F_hist[-1]
    T_now = T_hist[-1]

    now_lines = split_lines(T_now)
    now_chunks = chunk_lines(now_lines, chunk_size)

    if not now_chunks:
        return state

    new_F_hist = []
    new_T_hist = []

    beta_base = λ * erotic_retroaction_strength(state["erotic_vector"]) * agreement

    Lf = len(F_hist)
    Lt = len(T_hist)
    L = min(Lf, Lt)  # only rewrite where both exist

    for k in range(L):
        decay = np.exp(-(L - 1 - k) / tau)
        beta = beta_base * decay

        # -----------------------------
        # FEATURE REWRITE
        # -----------------------------
        F_past = F_hist[k]
        new_F_hist.append(F_past + beta * (F_now - F_past))

        # -----------------------------
        # POETIC MEMORY REWRITE
        # -----------------------------
        T_past = T_hist[k]
        past_lines = split_lines(T_past)
        past_chunks = chunk_lines(past_lines, chunk_size)

        if not past_chunks:
            new_T_hist.append(T_past)
            continue

        n_inject = max(1, int(beta * len(now_chunks)))
        incoming = now_chunks[-n_inject:]
        n_keep = max(1, len(past_chunks) - n_inject)

        revised_chunks = past_chunks[:n_keep] + incoming

        new_T_hist.append("\n".join(revised_chunks))

    # Preserve any extra feature states that have no text yet
    if Lf > L:
        new_F_hist.extend(F_hist[L:])

    # Preserve any extra text states (rare but safe)
    if Lt > L:
        new_T_hist.extend(T_hist[L:])

    print("History (features + poetic memory) rewritten!")

    state["feature_history"] = new_F_hist
    state["text_history"] = new_T_hist

    return state


In [30]:


def render_multimodal(state, text, audio_noise = None, video_noise = None, event = None):
    """
    Registers multimodal gestures (audio, video) into state.
    Separates static latent features vs dynamic gestures.
    
    Args:
        state: full system state
        text: negotiated / rendered text for this step
        audio_noise: dict of audio gesture features
        video_noise: dict of video gesture features
        event: the current NoiseEvent (optional, for reference)
    """

    # --------------------------------------------------
    # 0. Ensure history containers exist
    # --------------------------------------------------
    state.setdefault("multimodal_latent_history", {})
    state["multimodal_latent_history"].setdefault("audio", [])
    state["multimodal_latent_history"].setdefault("video", [])
    state["multimodal_latent_history"].setdefault("text", [])

    # --------------------------------------------------
    # 1. Text registration
    # --------------------------------------------------
    text_record = {
        "step": state.get("step", 0),
        "content": text,
        "event_ref": id(event) if event else None
    }
    state["multimodal_latent_history"]["text"].append(text_record)

    # --------------------------------------------------
    # 2. Audio gesture registration
    # --------------------------------------------------
    if audio_noise:
        audio_record = {
            "step": state.get("step", 0),
            "indices": audio_noise.get("indices", []),
            "intensity": audio_noise.get("intensity", 0.0),
            "rupture": audio_noise.get("rupture", 0.0),
            "jitter": audio_noise.get("jitter", 0.0),
            "event_ref": id(event) if event else None
        }
        state["multimodal_latent_history"]["audio"].append(audio_record)

    # --------------------------------------------------
    # 3. Video gesture registration
    # --------------------------------------------------
    if video_noise:
        video_record = {
            "step": state.get("step", 0),
            "indices": video_noise.get("indices", []),
            "intensity": video_noise.get("intensity", 0.0),
            "rupture": video_noise.get("rupture", 0.0),
            "jitter": video_noise.get("jitter", 0.0),
            "event_ref": id(event) if event else None
        }
        state["multimodal_latent_history"]["video"].append(video_record)

    # --------------------------------------------------
    # 4. Optional: link gesture history to the current NoiseEvent
    # --------------------------------------------------
    if event:
        event.gesture_history = {
            "audio": state["multimodal_latent_history"]["audio"][-1] if audio_noise else None,
            "video": state["multimodal_latent_history"]["video"][-1] if video_noise else None,
            "text": state["multimodal_latent_history"]["text"][-1]
        }

    return state


In [31]:

def call_writer_model(prompt, model):
    if model == "gpt-4.1":
        return call_gpt41(prompt)
    elif model == "qwen2.5-7b":
        return call_qwen(prompt)

def writer(prompt, model):
    return call_writer_model(prompt, model)


In [32]:
noise_signatures = noise_signature

In [33]:
cmds_list = []

import time

In [34]:

import re

def lexical_echo(a, b, min_overlap=0.12, ghost_weight=0.5):
    """
    Detects whether text `a` carries lexical residue of text `b`.

    Combines:
    • direct word overlap
    • soft/fragment overlap (for corrupted or partial echoes)

    ghost_weight controls how much the soft overlap contributes (0–1).
    """

    if not a or not b:
        return False

    def tokenize(t):
        t = t.lower()
        t = re.sub(r"[^a-z0-9\s]", " ", t)
        return [w for w in t.split() if len(w) > 3]

    A_list = tokenize(a)
    B_list = tokenize(b)

    if not A_list or not B_list:
        return False

    A = set(A_list)
    B = set(B_list)

    # --- 1. Direct lexical overlap ---
    direct_hits = len(A.intersection(B))
    direct_score = direct_hits / len(B)

    # --- 2. Soft / ghost overlap ---
    ghost_hits = 0
    for w in B:
        if any(w in a or a in w for a in A):
            ghost_hits += 1

    ghost_score = ghost_hits / len(B)

    # --- 3. Hybrid score ---
    combined_score = (1 - ghost_weight) * direct_score + ghost_weight * ghost_score

    return combined_score >= min_overlap


In [35]:

from sklearn.metrics.pairwise import cosine_similarity

def compute_shard_affinity(F_vec, shards):
    """
    Scalar in [0,1] representing how strongly shards resonate
    with the current textual field.
    """

    if not shards:
        return 0.0

    F_vec = np.asarray(F_vec, dtype=float).reshape(1, -1)
    affinities = []

    for s in shards:

        # ---------------- TEXT SHARDS ----------------
        if s["type"] == "text":
            # content = s.get("content", "") or s.get("original", "") 
            content = s.get("content", "")
            shard_vec = embed_text(content)

            if shard_vec is None:
                affinities.append(0.0)
                continue

            shard_vec = np.asarray(shard_vec, dtype=float).reshape(1, -1)

            # dimensional mismatch → weak resonance
            if shard_vec.shape[1] != F_vec.shape[1]:
                affinities.append(0.0)
                continue

            sim = cosine_similarity(F_vec, shard_vec)[0][0]

            # map from [-1,1] → [0,1]
            affinities.append((sim + 1.0) / 2.0)

        # ------------- AUDIO / VIDEO SHARDS ----------
        elif s["type"] in ("audio_gesture", "video_gesture"):
            content = s.get("meta", "")
            rr = extract_number(content, "'rupture_ratio'")
            rec = extract_number(content, "recursive_score")

            affinities.append(
                clamp01(0.5 * rr + 0.5 * normalize(rec))
            )

        else:
            affinities.append(0.0)

    return clamp01(sum(affinities) / len(affinities))


def estimate_shard_uptake(text, shards):
    if not shards:
        return 1.0

    signals = 0
    for s in shards:
        if s["type"] == "text":
            # if lexical_echo(text, s["original"] + "\n\n" + s.get("translation", ""), min_overlap=0.10, ghost_weight=0.6):
            if lexical_echo(text, s["content"], min_overlap=0.10, ghost_weight=0.6):
                signals += 1
        else:
            if modality_trace_detected(text, s["type"]):
                signals += 1

    return signals / len(shards)

def distort_shards(shards, noise_signature, intensity):
    distorted = []
    for s in shards:
        s2 = s.copy()
        s2["content"] = corrupt_text(
            s["content"],
            noise_signature,
            intensity=intensity
        )
        s2["source"] = f"ghost::{s.get('source','unknown')}"
        distorted.append(s2)
    return distorted


def modality_trace_detected(text, modality):
    # Always return False for now
    return False


import random
import string


def corrupt_text(text, noise_signature, intensity=1.0):
    if not text or not text.strip():
        return text  # nothing to corrupt

    words = text.split()

    if len(words) == 0:
        return text

    n = max(1, int(len(words) * 0.1 * intensity))
    n = min(n, len(words))  # never exceed word count

    for _ in range(n):
        i = random.randrange(len(words))  # safer than randint

        if random.random() < 0.5:
            # erode word
            w = words[i]
            words[i] = w[:max(1, len(w)//2)]
        else:
            # duplicate word fragment
            words.insert(i, words[i][:max(1, len(words[i])//2)])

    return " ".join(words)


In [36]:

def embed_text(text):
    F = extract_full_stanza_representation(text)
    F_vec, _ = flatten_feature_dict(F)
    return np.asarray(F_vec, dtype=float)


In [37]:

import random


def select_text_for_noise(state, text_shards):
    """
    Selects a textual residue for gesture/noise inference.
    Never returns empty string.
    """

    # 1. Filter the list first
    filtered_shards = [s['content'] for s in text_shards if s["type"] == "text"]

    # 2. Check if the list has any items before picking
    if filtered_shards:
        selected = random.choice(filtered_shards)
    else:
        selected = None

    return selected
    

In [38]:

import re

def extract_number(text, key):
    """
    Fuzzy numeric extractor from shard content.
    """

    if not text:
        return 0.0

    # Defensive normalization
    if isinstance(text, dict):
        text = str(text)

    key_variants = {
        "rupture": ["rupture", "rupture_ratio"],
        "recursion": ["recursion", "recursive_score"]
    }

    patterns = key_variants.get(key, [key])

    for p in patterns:
        match = re.search(rf"{p}[^0-9\-\.]*([0-9]+\.?[0-9]*)", text)
        if match:
            try:
                return float(match.group(1))
            except ValueError:
                continue

    return 0.0

def normalize(x, max_val=10.0):
    return clamp01(x / max_val)

In [39]:

def destabilize_text(text, state, shards, mode="temporal_scar"):
    if mode == "temporal_scar":
        injections = []

        # 1. random shard line
        for s in shards:
            if s["type"] == "text":
                # injections.append(extract_random_line(s["original"] + "\n\n" + s.get("translation", "")))
                injections.append(extract_random_line(s["content"]))
        
        # 2. ghost pressure
        if state.get("rag_ghosts"):
            injections.append("[ghost pressure intensifies]")

        if injections:
            return text + "\n\n" + random.choice(injections)

    return text


In [40]:

# UPDATED
def flatten_feature_dict(F):
    if not isinstance(F, dict):
        return np.zeros(14), "empty"

    audio = F.get("audio_features", {})
    affect = F.get("affect_vector", {})
    temporal = F.get("temporal_features", {})

    vec = np.array([
        audio.get("syllable_density", 0.0),     # 0
        audio.get("tempo", 0.0),                # 1
        audio.get("pacing_variance", 0.0),      # 2
        audio.get("enjambments", 0.0),           # 3
        audio.get("caesura", 0.0),               # 4
        audio.get("silence_ratio", 0.0),         # 5

        affect.get("valence", 0.0),              # 6
        affect.get("arousal", 0.0),              # 7
        affect.get("energy", 0.0),               # 8

        temporal.get("score_linear", 0.0),       # 9
        temporal.get("score_cyclical", 0.0),     # 10
        temporal.get("score_recursive", 0.0),    # 11
        temporal.get("score_hybrid", 0.0),       # 12
        temporal.get("number_of_motifs", 0.0),   # 13
    ], dtype=float)

    return vec, "structured_v1"


In [41]:

state = initialize_state_from_previous_stage(seed_state)

states = [state]

noise_signatures = noise_signatures

state.setdefault("rag_last", None)

# ----------------------------------------------------------
# TEMPORAL RECURSION (slow feedback scalar)
# ----------------------------------------------------------
state["temporal_recursion"] = seed_state.get("temporal_recursion", 0.2)


In [84]:
import copy

In [94]:

# ----------------------------------------------------------
# MAIN FIELD LOOP (RESUMABLE)
# ----------------------------------------------------------

def run_sympoietic_field(state, step_range):
    """
    step_range: iterable of step indices, e.g. range(0, 3) or range(5, 12)
    Assumes `state` already exists in scope (initialized or reloaded).
    """

    # states = []
    # noise_signatures = []

    for step in step_range:

        # Guard against accidental rewind
        if "step" in state and step < state["step"]:
            continue

        state["step"] = step

        # ---------------------------------------------
        # 0. Current textual field → F, F_vec
        # ---------------------------------------------
        current_text = state["text_history"][-1]

        F = extract_full_stanza_representation(current_text)
        F_vec, F_schema = flatten_feature_dict(F)

        # ---------------------------------------------
        # 1. Generate noise from current state; generate profile
        # ---------------------------------------------
        noise_signature = generate_noise_field(state)

        alpha = state.get("alpha", 0.5)
        profile = alpha_to_prompt_profile(alpha, state)

        # 1.1 Build NoiseEvent (base profile only)
        noise_profile = build_noise_profile_from_state(state)
        event = NoiseEvent(profile=noise_profile)

        # ---------------------------------------------
        # 2. Sample multimodal field once
        # ---------------------------------------------
        # λ = clamp01(state["desire"] + state["topography"] - state["stability"])
        λ = clamp01(
            state["desire"].get("arousal", 0.0) +
            state["desire"].get("excess", 0.0) +
            state.get("topography", 0.0) -
            state.get("stability", 0.5)
        )

        # --- TEXT RAG ---
        text_shards = sample_rag_with_noise(
            corpus=state["rag_memory"].get("text"),
            F_vec=F_vec,
            F_schema=F_schema,
            noise_signature=noise_signature,
            profile_history=state.get("profile_history", [])
        )

        # --- AUDIO ---
        audio_shards = sample_audio_corpus_with_noise(
            affect_audio_vectors=affect_audio_vectors,
            audio_analyses=audio_analyses,
            basic_audio_features=basic_audio_features,
            F_vec=F_vec,
            F_schema=F_schema,
            state=state,
            noise_signature=noise_signature
        )

        # --- VIDEO ---
        video_shards = sample_video_corpus_with_noise(
            affect_video_vectors=affect_video_vectors,
            video_analyses=video_analyses,
            basic_video_features=basic_video_features,
            F_vec=F_vec,
            F_schema=F_schema,
            state=state,
            noise_signature=noise_signature
        )

        shards = normalize_shards(text_shards, audio_shards, video_shards)

        # ---------------------------------------------
        # 2.01 Shard affinity → field pressure
        # ---------------------------------------------
        shard_affinity = compute_shard_affinity(F_vec, shards)
        state["last_shard_affinity"] = shard_affinity

        state.setdefault("shard_affinity_history", []).append({
            "step": step,
            "affinity": shard_affinity
        })

        # Shards curve alpha and destabilize equilibrium
        alpha = state.get("alpha", 0.5)

        alpha = clamp01(
            alpha
            + 0.25 * shard_affinity
            - 0.15 * state.get("stability", 0.5)
        )

        state["alpha"] = alpha

        # High shard pressure destabilizes voice dominance
        state["force_voice_instability"] = shard_affinity > 0.6

        # ---------------------------------------------
        # 2.03 Ghost shard reinjection (haunting)
        # ---------------------------------------------
        ghosts = state.get("rag_ghosts", [])

        if ghosts:
            ghost_shards = distort_shards(
                ghosts,
                noise_signature,
                intensity=0.5 + shard_affinity
            )
        else:
            ghost_shards = []

        # Fold ghosts back into the field
        if ghost_shards:
            shards = normalize_shards(
                text_shards,
                audio_shards,
                video_shards,
                ghosts=ghost_shards
            )

        # ---------------------------------------------
        # 2.04 Ghost pressure → forced instability
        # ---------------------------------------------
        if state.get("rag_ghosts"):
            state["force_voice_instability"] = True

        # ---------------------------------------------
        # 2.1 Initial gesture inference (provisional)
        # ---------------------------------------------

        text_for_noise = text_shards[0]['original'] if isinstance(text_shards, list) and text_shards else select_text_for_noise(state, text_shards)

        audio_noise = generate_audio_noise(
            event=event,
            text_noise=text_for_noise,
            audio_shards=audio_shards
        )

        video_noise = generate_video_noise(
            event=event,
            text_noise=text_for_noise,
            video_shards=video_shards
        )

        state["gesture_last"] = {
            "audio": audio_noise,
            "video": video_noise,
            "step": state["step"]
        }

        # ---------------------------------------------
        # 3. Render both agents
        # ---------------------------------------------
        alpha = state.get("alpha", 0.5)
        
        render_out = field_render(
            state=state,
            λ=λ,
            noise_field=noise_signature,
            shards=shards,
            alpha=state["alpha"]
        )

        voice = render_out["voice"]
        state["voice_history"].append(voice)
        
        text_out = render_out["text"]
        model_used = render_out["model"]

        voice_switch = (
            state.get("last_voice") is not None and
            voice != state["last_voice"]
        )

        state["voice_weights"] = render_out["voice_weights"]

        state["last_voice"] = voice

        # ---------------------------------------------
        # 4. Analyze outputs
        # ---------------------------------------------
        # poet_analysis = extract_profile_from_output(poet_out)
        # llm_analysis  = extract_profile_from_output(llm_out)
        analysis  = extract_profile_from_output(text_out)

        # merged_analyses = {
            # "density":   0.55 * poet_analysis["density"]   + 0.45 * llm_analysis["density"],
            # "recursion": 0.45 * poet_analysis["recursion"] + 0.55 * llm_analysis["recursion"],
            # "arousal":   0.60 * poet_analysis["arousal"]   + 0.40 * llm_analysis["arousal"],
            # "energy":    0.55 * poet_analysis["energy"]    + 0.45 * llm_analysis["energy"]
        #}

        # ---------------------------------------------
        # 4.1 Shard uptake estimation
        # ---------------------------------------------
        uptake_score = estimate_shard_uptake(text_out, shards)   # CONSIDER (OR TRY OUT) REPLACING THIS W text_only = [s for s in shards if s["type"] == "text"] uptake_score = estimate_shard_uptake(text_out, text_only)

        state.setdefault("shard_uptake_history", []).append({
            "step": step,
            "uptake": uptake_score,
            "voice": voice
        })

        # ---------------------------------------------
        # 4.2 Persist ignored shards as ghosts
        # ---------------------------------------------
        if uptake_score < 0.4:
            ignored = [s for s in shards if not s.get("source", "").startswith("ghost::")]

            if ignored:
                state.setdefault("rag_ghosts", []).extend(ignored)

        state["rag_ghosts"] = state.get("rag_ghosts", [])[-12:]

        # ---------------------------------------------
        # 5. Ontological feedback → profile_history
        # ---------------------------------------------
        # state.setdefault("profile_history", []).append(merged_analyses)
        state.setdefault("profile_history", []).append(analysis)

        # ---------------------------------------------
        # 6. Alpha evolves implicitly next iteration
        # ---------------------------------------------
        state["alpha"] = compute_alpha_with_history(state)

        # ---------------------------------------------
        # 7. Negotiated convergence
        # ---------------------------------------------
        # ---------------------------------------------
        # 3.1 Break fixed-point textual equilibrium
        # ---------------------------------------------
        if text_out.strip() == current_text.strip():
            # fracture = fracture_insert(state, text_shards)
            # text_out = text_out + "\n\n" + fracture
            state["frozen_steps"] = state.get("frozen_steps", 0) + 1
            text_out = destabilize_text(
                text_out,
                state=state,
                shards=shards,
                mode="temporal_scar"
            )
        else:
            state["frozen_steps"] = 0

        if state["frozen_steps"] >= 2:
            state["force_voice_instability"] = True
            state["alpha"] = clamp01(state["alpha"] + 0.2)
            
        if voice_switch:
            print("Voice switch! Negotiating outputted poems...")
            # check agreement and try and make the texts converge
            negotiated = converge(current_text, text_out, state, states, noise_signature, profile) # def converge(text_A, text_B, state, states, noise_field, profile, max_iter=4)
        else:
            negotiated = text_out

        # ---------------------------------------------
        # 7a. Enrich noise profile from negotiated text
        # ---------------------------------------------
        if not event.enriched:
            F_neg = extract_full_stanza_representation(negotiated)
            F_neg_vec, F_neg_schema = flatten_feature_dict(F_neg)

            event.profile = enrich_noise_profile(
                event.profile,
                F_neg_vec,
                F_neg_schema
            )
            event.enriched = True

        # ---------------------------------------------
        # 7b. Final gesture inference (authoritative)
        # ---------------------------------------------
        final_audio_noise = generate_audio_noise(
            event=event,
            text_noise=negotiated,
            audio_shards=audio_shards
        )

        final_video_noise = generate_video_noise(
            event=event,
            text_noise=negotiated,
            video_shards=video_shards
        )

        # RAG LAST UPDATE
        state["rag_last"] = {
            "text": text_shards,
            "audio": audio_shards,
            "video": video_shards,
            "links": state["rag_memory"].get("links", [])
        }

        # ---------------------------------------------
        # 7b.1 Ghost-induced gesture destabilization
        # ---------------------------------------------
        if state.get("rag_ghosts"):

            # Audio destabilization
            final_audio_noise["grain"] = (
                final_audio_noise.get("grain", 0.0) + 0.2
            )
            final_audio_noise["jitter"] = (
                final_audio_noise.get("jitter", 0.0) + 0.1
            )

            # Video destabilization
            final_video_noise["cut_rate"] = (
                final_video_noise.get("cut_rate", 0.0) + 0.15
            )
            final_video_noise["blur"] = (
                final_video_noise.get("blur", 0.0) + 0.1
            )

            ghost_pressure = min(0.3, 0.05 * len(state["rag_ghosts"]))
            state["desire"]["excess"] += ghost_pressure
            state["stability"] -= ghost_pressure

        # ---------------------------------------------
        # 7c. Gesture → state feedback
        # ---------------------------------------------
        state = apply_gesture_feedback(
            state,
            final_audio_noise,
            final_video_noise
        )

        # ---------------------------------------------
        # 7d. Noise-triggered intermedia rupture
        # ---------------------------------------------
        if should_trigger_links(state, noise_signature):
            links = sample_intermedia_links(
                state["rag_memory"].get("links", [])
            )

            if links:
                flaneur_note = (
                    "\n\n[FLÂNEUR NOTE]\n"
                    "Watch these video / intermedia poems and performance fragments "
                    "before continuing your flânerie:\n"
                )
                for l in links:
                    flaneur_note += f"– {l}\n"

                negotiated += flaneur_note

        # ---------------------------------------------
        # 8. Append history
        # ---------------------------------------------
        state["text_history"].append(negotiated)

        # ---------------------------------------------
        # 9. Temporal recursion
        # ---------------------------------------------
        state = temporal_retroaction_conditioned(state)

        # ---------------------------------------------
        # 9.1 Multimodal rendering
        # ---------------------------------------------
        render_multimodal(state, negotiated, audio_noise = final_audio_noise, video_noise = final_video_noise, event = event)

        # ---------------------------------------------
        # 10. Persist
        # ---------------------------------------------
        state.setdefault("meta", {})["last_completed_step"] = step

        states.append(copy.deepcopy(state))
        noise_signatures.append(noise_signature)

        with open(f"margento_hk_sympoiesis_states_{state['step']}.pkl", "wb") as f:
            pickle.dump(states, f)

        with open(f"margento_hk_sympoiesis_noise_logs_{state['step']}.pkl", "wb") as f:
            pickle.dump(noise_signatures, f)

        # --- EMIT FFMPEG COMMANDS ---
        total_chunks_audio = len(affect_audio_vectors[audio_shards['file']]['vectors'])
        total_chunks_video = len(affect_video_vectors[video_shards['file']]['vectors'])
        
        cmds = emit_ffmpeg_commands(event, final_video_noise, final_audio_noise, total_chunks=min(total_chunks_audio, total_chunks_video), output_prefix="sympoiesis", media_folder=MEDIA_FOLDER)

        # --- RUN FROM JUPYTER ---
        for c in cmds:
            print(c)
            !{c}

        cmds_list.append(cmds) 

        # ---------------------------------------------
        # 10.1 Update transient instability flags
        # ---------------------------------------------
        state["force_voice_instability"] *= 0.6   # decay instead of hard reset

        time.sleep(0.4)

    return states


In [ ]:

run_sympoietic_field(state, (0, 4))

In [34]:

import pickle

with open('margento_hk_sympoiesis_states_1.pkl', 'rb') as file:
    states = pickle.load(file)

In [ ]:

last = states[-1].get("step", -1) + 1

run_sympoietic_field(states[-1], range(last, last + 4))

In [42]:

import pickle

with open('margento_hk_sympoiesis_states_2.pkl', 'rb') as file1:
    states = pickle.load(file1)

In [ ]:
last = states[-1].get("step", -1) + 1

run_sympoietic_field(states[-1], range(last, last + 4))

In [109]:

import pickle

with open('margento_hk_sympoiesis_states_3.pkl', 'rb') as file:
    states = pickle.load(file)

In [ ]:

last = states[-1].get("step", -1) + 1

run_sympoietic_field(states[-1], range(last, last + 4))

In [99]:

with open('margento_hk_sympoiesis_states_63.pkl', 'rb') as file65:
    states = pickle.load(file65)

with open('margento_hk_sympoiesis_noise_logs_63.pkl', 'rb') as file64:
    noise_signatures = pickle.load(file64)

In [102]:


last = states[-1].get("step", -1) + 1

run_sympoietic_field(states[-1], range(last, last + 6))

Voice switch! Negotiating outputted poems...

=== AUDIO FILTER GRAPH ===
[0:a]asplit=7[a0s][a1s][a2s][a3s][a4s][a5s][a6s]; [a0s]atrim=start=44.8824:end=47.3758,asetpts=PTS-STARTPTS[aud0]; [a1s]atrim=start=107.219:end=109.7125,asetpts=PTS-STARTPTS[aud1]; [a2s]atrim=start=47.3758:end=49.8693,asetpts=PTS-STARTPTS[aud2]; [a3s]atrim=start=0.0:end=2.4935,asetpts=PTS-STARTPTS[aud3]; [a4s]atrim=start=52.3628:end=54.8562,asetpts=PTS-STARTPTS[aud4]; [a5s]atrim=start=84.7778:end=87.2713,asetpts=PTS-STARTPTS[aud5]; [a6s]atrim=start=47.3758:end=49.8693,asetpts=PTS-STARTPTS[aud6]; [aud0][aud1][aud2][aud3][aud4][aud5][aud6]concat=n=7:v=0:a=1[outa]


=== VIDEO FILTER GRAPH ===
[0:v]split=7[v0s][v1s][v2s][v3s][v4s][v5s][v6s]; [v0s]trim=start=109.8922:end=113.2223,setpts=PTS-STARTPTS[v0]; [v1s]trim=start=109.8922:end=113.2223,setpts=PTS-STARTPTS[v1]; [v2s]trim=start=16.6503:end=19.9804,setpts=PTS-STARTPTS[v2]; [v3s]trim=start=69.9314:end=73.2615,setpts=PTS-STARTPTS[v3]; [v4s]trim=start=0.0:end=3.3301,se

KeyboardInterrupt: 